In [ ]:
import pandas as pd
import numpy as np

from haversine import haversine_vector, Unit

In [ ]:
from haversine import haversine_vector, Unit
import numpy as np

# 一個起始點
start_point = (30.6574, 104.0658)  # 成都

# 多個目的地
destinations = [
    (29.5630, 106.5516), # 重慶
    (26.5878, 106.7072), # 貴陽
    (34.2648, 108.9441)  # 西安
]

# 使用 haversine_vector 計算距離矩陣
# 這會回傳一個 numpy 陣列，包含 start_point 到每一個 destination 的距離
distances_km = haversine_vector([start_point], destinations, Unit.KILOMETERS, comb=True)

print("起始點：", start_point)
print("目的地列表：", destinations)
print("\\n計算出的距離矩陣 (公里):")
print(distances_km)

# 如果您有兩組點，也可以計算它們之間的配對距離
# 使用字典來儲存地點，更具可讀性
points_A_dict = {
    "成都": (30.6574, 104.0658),
    "上海": (31.2304, 121.4737)
}

points_B_dict = {
    "重慶": (29.5630, 106.5516),
    "貴陽": (26.5878, 106.7072),
    "西安": (34.2648, 108.9441)
}

# 從字典中提取座標列表和名稱列表
points_A_coords = list(points_A_dict.values())
points_A_names = list(points_A_dict.keys())

points_B_coords = list(points_B_dict.values())
points_B_names = list(points_B_dict.keys())

# 計算距離矩陣
distances_matrix = haversine_vector(points_A_coords, points_B_coords, Unit.KILOMETERS, comb=True)

# 建立 DataFrame，並使用字典的鍵作為 index 和 columns
distances_matrix_df = pd.DataFrame(distances_matrix, index=points_B_names, columns=points_A_names).round(2)
distances_matrix_df

In [ ]:
cd_more_one_df = pd.read_csv("成都_strategy_A_multiple_stations_per_cpo.csv")
cd_emo_df = pd.read_csv("成都_public_competitor_stations.csv")
cd_emo_df.rename(columns={"name": "station_name"}, inplace=True)
cd_emo_df = cd_emo_df[cd_emo_df["cpo_final"] != "蔚来"].reset_index(drop=True)

In [ ]:
cd_more_one_df[["station_name", "longitude", "latitude"]]
cd_emo_df[["station_name", "longitude", "latitude"]]

In [ ]:
# 1. 從 DataFrame 中提取座標和名稱
points_A_coords = list(zip(cd_more_one_df['latitude'], cd_more_one_df['longitude']))
points_A_names = cd_more_one_df['station_name'].tolist()

points_B_coords = list(zip(cd_emo_df['latitude'], cd_emo_df['longitude']))
points_B_names = cd_emo_df['station_name'].tolist()

# 2. 計算距離矩陣
# haversine_vector 返回的矩陣形狀為 (len(points_A), len(points_B))
distance_matrix_ab = haversine_vector(points_A_coords, points_B_coords, Unit.KILOMETERS, comb=True)

# 3. 建立 DataFrame
# index 應對應 points_A, columns 應對應 points_B
distance_df = pd.DataFrame(distance_matrix_ab, index=points_B_names, columns=points_A_names).round(2)

distance_df.min(), distance_df.idxmin()

In [ ]:
# 筛选出包含“理想”的行
li_auto_df = distance_df[distance_df.index.str.contains('理想')]

# 筛选出包含“小鹏”的行
xpeng_df = distance_df[distance_df.index.str.contains('小鹏')]

# 找到每个原始站点到“理想”品牌的最近距离
closest_li_auto = li_auto_df.min()

# 找到每个原始站点到“小鹏”品牌的最近距离
closest_xpeng = xpeng_df.min()

print("--- 最近的理想站点距离 ---")
print(closest_li_auto.sort_values().head())

print("\n--- 最近的小鹏站点距离 ---")
print(closest_xpeng.sort_values().head())

In [ ]:
# --- 执行替换操作 ---

# 1. 找出要替换的原始站点名称
station_to_replace_for_li = '愉秒充新园大道充电站'
station_to_replace_for_xpeng = '昊铂成都王府井百货总府路店超充站'

# 2. 找到对应的最佳OME站点名称
# idxmin() 会找到每列（每个原始站点）的最小值对应的索引（OME站点名）
li_replacement_name = li_auto_df.idxmin()[station_to_replace_for_li]
xpeng_replacement_name = xpeng_df.idxmin()[station_to_replace_for_xpeng]

print(f"'{station_to_replace_for_li}' 将被替换为理想站点: '{li_replacement_name}'")
print(f"'{station_to_replace_for_xpeng}' 将被替换为小鹏站点: '{xpeng_replacement_name}'")

# 3. 从OME数据表 (cd_emo_df) 中获取这两个新站点的完整信息
li_station_data = cd_emo_df[cd_emo_df['station_name'] == li_replacement_name]
xpeng_station_data = cd_emo_df[cd_emo_df['station_name'] == xpeng_replacement_name]

# 4. 创建最终的目标列表副本
final_target_df = cd_more_one_df.copy()

# 5. 删除旧的站点
final_target_df = final_target_df[~final_target_df['station_name'].isin([station_to_replace_for_li, station_to_replace_for_xpeng])]

# 6. 合并新站点
# 注意：我们需要确保列名一致，cd_emo_df 可能缺少一些列，合并时会自动用 NaN 填充
final_target_df = pd.concat([final_target_df, li_station_data, xpeng_station_data], ignore_index=True)

# 7. 显示并保存最终名单
print("\n--- 更新后的目标清单 ---")
display(final_target_df)

# 保存到新的CSV文件
final_target_df.to_csv("final_target_list_CD.csv", index=False, encoding='utf-8-sig')
print("\n最终名单已保存到 'final_target_list_CD.csv'")

In [ ]:
# --- 在路线计划中直接替换并更新坐标 ---

# 1. 建立明确的替换映射
replacement_map = {
    station_to_replace_for_li: li_replacement_name,
    station_to_replace_for_xpeng: xpeng_replacement_name
}

# 2. 建立新站点的坐标映射
coords_map = {
    li_replacement_name: (li_station_data.iloc[0]['latitude'], li_station_data.iloc[0]['longitude']),
    xpeng_replacement_name: (xpeng_station_data.iloc[0]['latitude'], xpeng_station_data.iloc[0]['longitude'])
}

print("--- 站点替换关系 ---")
print(replacement_map)
print("\n--- 新站点坐标 ---")
print(coords_map)

# 3. 加载原始路线文件
route_df = pd.read_csv("output/report_CD_enriched.csv")

# 4. 循环执行替换
for old_name, new_name in replacement_map.items():
    # 获取新站点的坐标
    new_lat, new_lon = coords_map[new_name]
    
    # 找到所有需要替换的行
    rows_to_update = route_df['目的地'] == old_name
    
    # 替换名称
    route_df.loc[rows_to_update, '目的地'] = new_name
    
    # 更新坐标
    route_df.loc[rows_to_update, '目的地緯度'] = new_lat
    route_df.loc[rows_to_update, '目的地經度'] = new_lon

# 5. 显示并保存更新后的路线
print("\n--- 更新后的路线计划 (部分预览) ---")
# 筛选出被修改的行和它们前后的记录，以供查验
display(route_df[route_df['目的地'].isin(replacement_map.values())])

route_df.to_csv("output/report_CD_replaced.csv", index=False, encoding='utf-8-sig')
print("\n已将更新后的完整路线保存到 'output/report_CD_replaced.csv'")

## CPO 統計

In [ ]:
import pandas as pd

# Load the station data for Chengdu
cd_stations_df = pd.read_csv('output/all_map_stations_CD.csv')

# Get the unique operator names
unique_cpos = cd_stations_df['operator_name'].unique()

# Print the list of unique CPOs
print("成都地区所有不重复的CPO列表:")
for cpo in sorted(unique_cpos):
    print(f"- {cpo}")

print(f"\n总计: {len(unique_cpos)} 个不重复的CPO")

In [ ]:
pd.DataFrame(unique_cpos)

In [ ]:
import pandas as pd
import glob

# 使用 glob 找到 Day/day1 和 Day/day2 下的所有 mission_test_records.csv
log_files = glob.glob('Day/day*/mission_test_records.csv')

# 檢查是否找到了檔案
if not log_files:
    print("錯誤：在 'Day/day1/' 或 'Day/day2/' 目錄下沒有找到 'mission_test_records.csv' 檔案。")
else:
    # 讀取所有日誌檔案並將它們合併成一個 DataFrame
    all_logs_df = pd.concat((pd.read_csv(f) for f in log_files), ignore_index=True)

    # 從 'CPO Name' 欄位中獲取不重複的 CPO 名稱
    # 使用 .dropna() 來忽略可能存在的空值
    tested_cpos = all_logs_df['CPO Name'].dropna().unique()

    # 打印出已測試的 CPO 列表
    print("Day 1 & Day 2 已測試的所有不重複的CPO列表:")
    for cpo in sorted(tested_cpos):
        print(f"- {cpo}")

    print(f"\n總計: {len(tested_cpos)} 個不重複的CPO")

In [ ]:
import pandas as pd
import glob
import numpy as np

# --- 步骤 1: 获取成都地区所有的 CPO 列表 ---
all_stations_df = pd.read_csv('output/all_map_stations_CD.csv')
all_cpos_series = all_stations_df['operator_name'].dropna().unique()
# 排除 'test' CPO (不区分大小写)
all_cpos_list = [cpo for cpo in all_cpos_series if 'test' not in cpo.lower()]


# --- 步骤 2: 获取 Day 1 & Day 2 已测试过的 CPO 列表 ---
log_files = glob.glob('Day/day*/mission_test_records.csv')

if not log_files:
    print("错误：在 'Day/day*/' 目录下没有找到 'mission_test_records.csv' 檔案。")
    tested_cpos_list = []
else:
    all_logs_df = pd.concat((pd.read_csv(f) for f in log_files), ignore_index=True)
    tested_cpos_series = all_logs_df['CPO Name'].dropna().unique()
    # 排除 'test' CPO (不区分大小写)
    tested_cpos_list = [cpo for cpo in tested_cpos_series if 'test' not in cpo.lower()]


# --- 步骤 3: 创建对比表格 ---
# 创建一个 DataFrame 来进行对比
comparison_df = pd.DataFrame(sorted(all_cpos_list), columns=['CPO Name'])

# 标记已访问的 CPO
comparison_df['Visited'] = np.where(comparison_df['CPO Name'].isin(tested_cpos_list), '✅', '❌')


# --- 步骤 4: 显示结果 ---
print("成都地区 CPO 任务完成情况一览表:\n")
# 使用 to_string() 来确保所有行都被打印出来
print(comparison_df.to_string())

# 分别打印已完成和未完成的列表
print("\n" + "="*40)
print("\n✅ 已完成的 CPO 列表:")
visited_df = comparison_df[comparison_df['Visited'] == '✅']
for cpo in visited_df['CPO Name']:
    print(f"- {cpo}")
print(f"\n已完成数量: {len(visited_df)}")


print("\n" + "="*40)
print("\n❌ 待完成的 CPO 列表:")
not_visited_df = comparison_df[comparison_df['Visited'] == '❌']
for cpo in not_visited_df['CPO Name']:
    print(f"- {cpo}")
print(f"\n未完成数量: {len(not_visited_df)}")

In [ ]:
import pandas as pd
import glob
import numpy as np
from pypinyin import pinyin, Style

# --- 准备工作：定义一个函数，将汉字字符串转换为无声调的连续拼音 ---
def to_pinyin_flat(text):
    """将 '太能衝' 转换为 'tainengchong'"""
    # style=Style.NORMAL 表示无声调
    pinyin_list = pinyin(text, style=Style.NORMAL)
    # 将 [['tai'], ['neng'], ['chong']] 转换为 'tainengchong'
    return ''.join([item[0] for item in pinyin_list])

# --- 步骤 1: 获取成都地区所有的 CPO 列表 ---
all_stations_df = pd.read_csv('output/all_map_stations_CD.csv')
all_cpos_series = all_stations_df['operator_name'].dropna().unique()
all_cpos_list_orig = [cpo for cpo in all_cpos_series if 'test' not in str(cpo).lower()]
# 创建一个拼音版的列表用于比对
all_cpos_list_pinyin = [to_pinyin_flat(cpo) for cpo in all_cpos_list_orig]

# --- 步骤 2: 获取 Day 1 & Day 2 已测试过的 CPO 列表 ---
log_files = glob.glob('Day/day*/mission_test_records.csv')

if not log_files:
    print("错误：在 'Day/day*/' 目录下没有找到 'mission_test_records.csv' 檔案。")
    tested_cpos_list_pinyin = []
else:
    all_logs_df = pd.concat((pd.read_csv(f) for f in log_files), ignore_index=True)
    tested_cpos_series = all_logs_df['CPO Name'].dropna().unique()
    tested_cpos_list_orig = [cpo for cpo in tested_cpos_series if 'test' not in str(cpo).lower()]
    # 创建一个拼音版的列表用于比对
    tested_cpos_list_pinyin = [to_pinyin_flat(cpo) for cpo in tested_cpos_list_orig]

# --- 步骤 3: 创建对比表格 ---
# 使用原始名称创建 DataFrame
comparison_df = pd.DataFrame(sorted(all_cpos_list_orig), columns=['CPO Name'])
# 创建一个临时的拼音列用于比对
comparison_df['CPO Name Pinyin'] = comparison_df['CPO Name'].apply(to_pinyin_flat)

# 使用拼音列表进行 'isin' 判断
comparison_df['Visited'] = np.where(comparison_df['CPO Name Pinyin'].isin(tested_cpos_list_pinyin), '✅', '❌')

# 删除临时的拼音列
comparison_df.drop(columns=['CPO Name Pinyin'], inplace=True)

# --- 步骤 4: 显示结果 ---
print("成都地区 CPO 任务完成情况一览表 (已处理拼音差异):\n")
print(comparison_df.to_string())

# --- 步骤 5: 打印去重后的已完成和未完成列表 ---
# 使用集合操作来获取准确的已完成和未完成的拼音CPO列表
all_cpos_set_pinyin = set(all_cpos_list_pinyin)
tested_cpos_set_pinyin = set(tested_cpos_list_pinyin)

visited_pinyin = sorted(list(tested_cpos_set_pinyin))
not_visited_pinyin = sorted(list(all_cpos_set_pinyin - tested_cpos_set_pinyin))

# 为了显示更友好的原始名称，我们从拼音反向查找一个原始名称
pinyin_to_orig_map_all = {to_pinyin_flat(name): name for name in reversed(all_cpos_list_orig)}
pinyin_to_orig_map_tested = {to_pinyin_flat(name): name for name in reversed(tested_cpos_list_orig)}


print("\n" + "="*40)
print("\n✅ 已完成的 CPO 列表 (按拼音去重):")
for p in visited_pinyin:
    # 优先从测试过的列表里找原始名，找不到再从总列表里找
    display_name = pinyin_to_orig_map_tested.get(p, pinyin_to_orig_map_all.get(p, p))
    print(f"- {display_name}")
print(f"\n已完成数量: {len(visited_pinyin)}")

print("\n" + "="*40)
print("\n❌ 待完成的 CPO 列表 (按拼音去重):")
for p in not_visited_pinyin:
    display_name = pinyin_to_orig_map_all.get(p, p)
    print(f"- {display_name}")
print(f"\n未完成数量: {len(not_visited_pinyin)}")

In [ ]:
# 1. 定义在成都已经完成测试的 CPO 列表 (根据您的输入)
chengdu_completed_cpo = [
    "小桔充电", "逸安启", "蔚景云", "云快充", "星星充电", "特来电", 
    "中装天如", "太能充", "太空充电", "壳牌充电", "极氪能源", "达克云", 
    "港投中油", "国家电网", "广汽能源", "蔚来", "百源智联", "蜀道新能源", 
    "南方電網", "新電徒", "昊柏", "领充创享", "亿谦电子", "特斯拉", 
    "億天宸", "公交新能源", "城頭能源", "小鵬", "崑崙網電", "惠知電", 
    "万净极速充", "道畅充充", "理想超充", "武候啇程", "開麥斯", "嵐圖", 
    "快鰻", "四川明星新能源"
]

# 假设 cq_cpo_list 已经从上一个单元格成功加载
# 为了代码的独立性，我们再获取一次
try:
    cq_stations_df = pd.read_csv('output/all_map_stations_CQ.csv')
    cq_cpo_list = cq_stations_df['operator_name'].unique()
except NameError:
    print("错误：请先运行上一个单元格来加载 'cq_cpo_list'。")
    # 或者在这里重新加载
    # cq_stations_df = pd.read_csv('output/all_map_stations_CQ.csv')
    # cq_cpo_list = cq_stations_df['cpo'].unique()


# 2. 转换为集合以方便计算
chengdu_set = set(chengdu_completed_cpo)
chongqing_set = set(cq_cpo_list)

# 3. 计算交集和差集
already_tested_in_cq = chongqing_set.intersection(chengdu_set)
new_targets_in_cq = chongqing_set.difference(chengdu_set)

# 4. 打印结果
print("--- 重庆 CPO 测试计划 ---")
print(f"总计: {len(chongqing_set)} 个 CPO")
print(f"已在成都测试过: {len(already_tested_in_cq)} 个")
print(f"新的测试目标: {len(new_targets_in_cq)} 个")
print("\\n" + "="*30 + "\\n")

print("✅ 新的测试目标列表 (重庆):")
if not new_targets_in_cq:
    print("太棒了！所有重庆的CPO似乎都已经在成都测试过。")
else:
    for cpo in sorted(list(new_targets_in_cq)):
        print(f"- {cpo}")

print("\\n" + "="*30 + "\\n")

print("📋 已在成都测试过的 CPO 列表 (重庆也有):")
if not already_tested_in_cq:
    print("无。")
else:
    for cpo in sorted(list(already_tested_in_cq)):
        print(f"- {cpo}")


In [ ]:
import pandas as pd
from pypinyin import pinyin, Style

# --- 准备工作 ---
def to_pinyin_flat(text):
    """将 '太能衝' 转换为 'tainengchong'"""
    if not isinstance(text, str):
        return ""
    pinyin_list = pinyin(text, style=Style.NORMAL)
    return ''.join([item[0] for item in pinyin_list])

# --- 1. 定义 CPO 列表 ---

# 在成都已经完成测试的 CPO 列表
chengdu_completed_cpo = [
    "小桔充电", "逸安启", "蔚景云", "云快充", "星星充电", "特来电", 
    "中装天如", "太能充", "太空充电", "壳牌充电", "极氪能源", "达克云", 
    "港投中油", "国家电网", "广汽能源", "蔚来", "百源智联", "蜀道新能源", 
    "南方電網", "新電徒", "昊柏", "领充创享", "亿谦电子", "特斯拉", 
    "億天宸", "公交新能源", "城頭能源", "小鵬", "崑崙網電", "惠知電", 
    "万净极速充", "道畅充充", "理想超充", "武候啇程", "開麥斯", "嵐圖", 
    "快鰻", "四川明星新能源"
]

# 重庆的 CPO 列表
chongqing_cpo_list = [
    "小桔充电", "广汽能源", "智道", "星星充电", "润诚达", "逸安启", 
    "幸福千万家", "爱众云鲲", "百源智联", "云快充", "极氪能源", "蔚景云", 
    "金弦新能源", "驿满充电", "滨南时代", "重庆交运", "愉秒充", "特瓦特", 
    "能银链", "重庆中盾", "重庆能投", "朗新新能源", "重庆壳牌", 
    "重庆小马爱冲", "易安桩", "国家电网", "成都亿嘉电", "易安充", "渝快充", 
    "达克云", "开鑫充", "开鑫充电", "特来电", "智泊惠", "一码YiMa", 
    "南网电动", "库仑", "吉智能源", "小飞象"
]

# --- 2. 转换为拼音集合 ---
chengdu_pinyin_set = {to_pinyin_flat(cpo) for cpo in chengdu_completed_cpo}
chongqing_pinyin_set = {to_pinyin_flat(cpo) for cpo in chongqing_cpo_list}

# --- 3. 创建从拼音到原始名称的映射 ---
# 这有助于我们显示用户友好的中文名称
pinyin_to_orig_map_cq = {to_pinyin_flat(name): name for name in reversed(chongqing_cpo_list)}

# --- 4. 计算差集 (新的测试目标) ---
new_targets_pinyin = chongqing_pinyin_set - chengdu_pinyin_set
already_tested_pinyin = chongqing_pinyin_set.intersection(chengdu_pinyin_set)

# --- 5. 打印结果 ---
print("--- 重庆 CPO 测试计划 (基于拼音精确比对) ---")
print(f"重庆 CPO 总数 (去重后): {len(chongqing_pinyin_set)} 个")
print(f"已在成都测试过: {len(already_tested_pinyin)} 个")
print(f"新的测试目标: {len(new_targets_pinyin)} 个")
print("\n" + "="*40 + "\n")

print("✅ 新的测试目标列表 (重庆):")
if not new_targets_pinyin:
    print("太棒了！所有重庆的CPO似乎都已经在成都测试过。")
else:
    # 从映射中找回原始中文名并排序
    new_targets_orig = sorted([pinyin_to_orig_map_cq[p] for p in new_targets_pinyin])
    for cpo in new_targets_orig:
        print(f"- {cpo}")

print("\n" + "="*40 + "\n")

print("📋 已在成都测试过的 CPO 列表 (重庆也有):")
if not already_tested_pinyin:
    print("无。")
else:
    already_tested_orig = sorted([pinyin_to_orig_map_cq[p] for p in already_tested_pinyin])
    for cpo in already_tested_orig:
        print(f"- {cpo}")


In [ ]:
# /Users/QXZ6BKI/work/logbook_deployment/output/final_report_GY.csv

import pandas as pd

gy_df = pd.read_csv('/Users/QXZ6BKI/work/logbook_deployment/output/all_map_stations_GY.csv')
gy_df = gy_df[gy_df["operator_name"].isin(['贵州云谷', '户户充', '南网电动', '聚合快充', '贵州畅的'])]
gy_df[gy_df['operator_name'] == "聚合快充"]

## 整理 CPO LISST

In [ ]:
import pandas as pd

cpo_df = pd.read_excel('output/Logbook_2026.xlsx')
cpo_df.head()

In [ ]:
import pandas as pd
import os

# 定义日志文件名
logbook_file = 'output/Logbook_2026.xlsx'

try:
    # --- 1. 读取整合后的日志文件 ---
    print(f"正在读取日志文件: {logbook_file} ...")
    log_df = pd.read_excel(logbook_file)

    # --- 2. 提取、去重、排序 CPO 名称 ---
    # 使用 .dropna() 清除空值，.unique() 获取唯一值
    cpo_names_cn = log_df['CPO Name'].dropna().unique()
    # 排序以获得一致的顺序
    cpo_names_cn_sorted = sorted(cpo_names_cn)

    print("\\n" + "="*40)
    print("从日志中提取到的所有不重复 CPO 名称:")
    for name in cpo_names_cn_sorted:
        print(f"- {name}")
    print(f"\\n总计: {len(cpo_names_cn_sorted)} 个")
    print("="*40 + "\\n")

    # --- 3. 创建 CPO 主列表文件 ---
    master_list_file = 'cpo_master_list.csv'
    
    # 创建一个 DataFrame
    master_df = pd.DataFrame({
        'cpo_name_cn': cpo_names_cn_sorted,
        'cpo_name_en': ''  # 创建一个空的英文名列，待填充
    })

    # 保存为 CSV 文件
    master_df.to_csv(master_list_file, index=False, encoding='utf-8-sig')
    
    print(f"✅ 成功创建 CPO 主列表文件: '{master_list_file}'")
    print("下一步，请打开这个 CSV 文件，为每一个中文名填写对应的英文名。")

except FileNotFoundError:
    print(f"❌ 错误：找不到文件 '{logbook_file}'。")
    print("请确认文件名是否正确，以及文件是否位于项目根目录下。")
except KeyError:
    print(f"❌ 错误：在文件 '{logbook_file}' 中找不到名为 'CPO Name' 的列。")
    print("请检查列名是否正确。")
except Exception as e:
    print(f"处理文件时发生未知错误: {e}")


In [ ]:
!pip install opencc-python-reimplemented pypinyin

In [ ]:

# 安装必要的库
import pandas as pd
import opencc
from pypinyin import pinyin, Style
import os

# --- 准备工作 ---

# 1. 初始化简繁转换器
cc_s2t = opencc.OpenCC('s2t')  # 简体到繁体
cc_t2s = opencc.OpenCC('t2s')  # 繁体到简体

# 2. 定义拼音转换函数
def to_pinyin_flat(text):
    """将汉字字符串转换为无声调的、以下划线连接的拼音"""
    if not isinstance(text, str):
        return ""
    # 先统一转为简体再生成拼音，确保一致性
    simplified_text = cc_t2s.convert(text)
    pinyin_list = pinyin(simplified_text, style=Style.NORMAL)
    # 将 [['zhong'], ['guo']] 转换为 'zhong_guo'
    return '_'.join([item[0] for item in pinyin_list if item[0]])

# --- 数据处理 ---

# 定义日志文件名
logbook_file = 'output/Logbook_2026.xlsx'

try:
    # 1. 读取整合后的日志文件
    print(f"正在读取日志文件: {logbook_file} ...")
    log_df = pd.read_excel(logbook_file)

    # 2. 提取、去重 CPO 名称
    cpo_names_orig = log_df['CPO Name'].dropna().unique()

    # 3. 创建包含三列数据的列表
    master_list_data = []
    for name in cpo_names_orig:
        simplified_name = cc_t2s.convert(name)
        traditional_name = cc_s2t.convert(name)
        pinyin_name = to_pinyin_flat(simplified_name)
        master_list_data.append({
            'cpo_name_cn_t': traditional_name,
            'cpo_name_cn_s': simplified_name,
            'cpo_name_en': pinyin_name
        })
    
    # 4. 创建 DataFrame 并去重
    master_df = pd.DataFrame(master_list_data)
    # 基于拼音列去重，因为不同写法可能指向同一个 CPO
    master_df.drop_duplicates(subset=['cpo_name_en'], inplace=True)
    # 按拼音排序
    master_df.sort_values(by='cpo_name_en', inplace=True)
    master_df.reset_index(drop=True, inplace=True)

    # --- 结果展示与保存 ---
    
    print("\n" + "="*50)
    print("CPO 主列表预览:")
    # 使用 display() 以获得更好的表格格式
    display(master_df)
    print("="*50 + "\n")

    # 5. 保存为 CSV 文件
    master_list_file = 'cpo_master_list.csv'
    master_df.to_csv(master_list_file, index=False, encoding='utf-8-sig')
    
    print(f"✅ 成功创建并写入 CPO 主列表文件: '{master_list_file}'")
    print("您可以检查文件内容，如果需要，可以手动调整英文名。")

except FileNotFoundError:
    print(f"❌ 错误：找不到文件 '{logbook_file}'。")
    print("请确认文件名是否正确，以及文件是否位于项目根目录下。")
except KeyError:
    print(f"❌ 错误：在文件 '{logbook_file}' 中找不到名为 'CPO Name' 的列。")
    print("请检查列名是否正确。")
except Exception as e:
    print(f"处理文件时发生未知错误: {e}")


## 城市重要性 PTPI

In [ ]:
import pandas as pd

car_counts_file = 'car_counts_city.csv'
national_charging_data_file = 'national_charge_station.csv'

car_counts_df = pd.read_csv(car_counts_file)
chargeing_station_df = pd.read_csv(national_charging_data_file)

In [ ]:

# 使用您提供的更准确的新能源车保有量数据
new_car_counts_data = {
    'city': ['成都市', '重庆市', '贵阳市'],
    'unique_vehicle_count': [1080000, 952000, 200000]
}
car_counts_target_city_df = pd.DataFrame(new_car_counts_data)

print("--- 使用更新后的新能源车保有量数据 ---")
display(car_counts_target_city_df)


In [ ]:
chargeing_station_target_city_df = chargeing_station_df[chargeing_station_df["city"].isin(city_list)].reset_index(drop=True)
chargeing_station_counts_df = chargeing_station_target_city_df["city"].value_counts().to_frame().reset_index().rename(columns={"count": "station_counts"})
chargeing_station_counts_df

In [ ]:
cpo_counts_df = chargeing_station_target_city_df.groupby("city")["operator_name"].nunique().to_frame().reset_index().rename(columns={"operator_name": "CPO_counts"})
cpo_counts_df

In [ ]:

# --- 步骤 4: 合并数据 ---
# 使用 pd.merge 来合并数据
merged_df = pd.merge(car_counts_target_city_df, chargeing_station_counts_df, on='city')
merged_df = pd.merge(merged_df, cpo_counts_df, on='city')

# --- 步骤 5: 定义权重并计算 PTPI ---
w1 = 0.3  # 车辆数权重
w2 = 0.4  # CPO数权重
w3 = 0.3  # 充电站数权重

merged_df['PTPI'] = (merged_df['unique_vehicle_count'] * w1 + 
                     merged_df['CPO_counts'] * w2 + 
                     merged_df['station_counts'] * w3)

# --- 步骤 6: 计算全国总数以供对比 ---
# 使用您提供的更准确的全国总数
total_vehicle_count = 36890000 
total_station_count = chargeing_station_df.shape[0]
# total_vehicle_count = car_counts_df['unique_vehicle_count'].sum()
total_station_count = chargeing_station_df.shape[0]
total_cpo_count = chargeing_station_df['operator_name'].nunique()

# 将城市数据与全国数据合并，以便于比较
merged_df['vehicle_percentage'] = (merged_df['unique_vehicle_count'] / total_vehicle_count) * 100
merged_df['station_percentage'] = (merged_df['station_counts'] / total_station_count) * 100
merged_df['cpo_percentage'] = (merged_df['CPO_counts'] / total_cpo_count) * 100


# --- 步骤 7: 显示最终结果 ---
# 重新排列列的顺序以便查看
final_ptpi_df = merged_df[[
    'city', 
    'PTPI',
    'unique_vehicle_count', 
    'vehicle_percentage',
    'station_counts',
    'station_percentage',
    'CPO_counts',
    'cpo_percentage'
]].sort_values(by='PTPI', ascending=False).reset_index(drop=True)

print("--- 城市选择重要系数 (PTPI) 计算结果 ---")
print(f"权重: 车辆数={w1}, CPO数={w2}, 充电站数={w3}\\n")

# 格式化输出，使百分比更易读
final_ptpi_df['vehicle_percentage'] = final_ptpi_df['vehicle_percentage'].map('{:.2f}%'.format)
final_ptpi_df['station_percentage'] = final_ptpi_df['station_percentage'].map('{:.2f}%'.format)
final_ptpi_df['cpo_percentage'] = final_ptpi_df['cpo_percentage'].map('{:.2f}%'.format)


display(final_ptpi_df)

print("\\n--- 全国数据总览 ---")
print(f"全国总车辆数: {total_vehicle_count}")
print(f"全国总充电站数: {total_station_count}")
print(f"全国总CPO数: {total_cpo_count}")


In [ ]:

# --- 步骤 8: 计算三个城市合并后的 CPO 覆盖率 ---

# 1. 从之前筛选出的目标城市充电站数据中，获取每个城市的 CPO 集合
cpo_cd_set = set(chargeing_station_target_city_df[chargeing_station_target_city_df['city'] == '成都市']['operator_name'].unique())
cpo_cq_set = set(chargeing_station_target_city_df[chargeing_station_target_city_df['city'] == '重庆市']['operator_name'].unique())
cpo_gy_set = set(chargeing_station_target_city_df[chargeing_station_target_city_df['city'] == '贵阳市']['operator_name'].unique())

# 2. 将三个城市的 CPO 集合合并，得到一个总的、不重复的 CPO 集合
combined_cpo_set = cpo_cd_set.union(cpo_cq_set).union(cpo_gy_set)
combined_cpo_count = len(combined_cpo_set)

# 3. 计算覆盖率 (total_cpo_count 来自上一个单元格的计算结果)
try:
    coverage_percentage = (combined_cpo_count / total_cpo_count) * 100
    
    print("--- 成都、重庆、贵阳三城 CPO 覆盖率分析 ---")
    print(f"成都、重庆、贵阳三地合计拥有不重复的 CPO 数量: {combined_cpo_count} 个")
    print(f"全国总 CPO 数量: {total_cpo_count} 个")
    print(f"三城合计覆盖了全国 CPO 的: {coverage_percentage:.2f}%")
    
except NameError:
    print("❌ 错误: 'total_cpo_count' 未定义。请先确保上一个计算 PTPI 的单元格已成功运行。")



## 城市熱力圖

## 城市充电站热力图与已访问站点标注

In [ ]:

import pandas as pd
from pyecharts import options as opts
from pyecharts.charts import Geo
from pyecharts.globals import ChartType

def create_city_heatmap_with_visited_stations(city_name, all_stations_df, visited_stations_df, amap_api_key):
    """
    创建城市充电站热力图，并在图上标注已访问的站点。

    :param city_name: 城市名称 (例如: "成都")
    :param all_stations_df: 包含所有充电站的 DataFrame，需要有 'longitude' 和 'latitude' 列。
    :param visited_stations_df: 包含已访问充电站的 DataFrame，需要有 'longitude' 和 'latitude' 列。
    :param amap_api_key: 您的高德地图 API Key.
    :return: pyecharts.charts.Geo 对象
    """
    
    # 准备热力图数据：[[lng, lat, 1], [lng, lat, 1], ...]
    heatmap_data = [[row['longitude'], row['latitude'], 1] for index, row in all_stations_df.iterrows()]
    
    # 准备已访问站点数据
    visited_points = [(row['station_name'], row['longitude'], row['latitude']) for index, row in visited_stations_df.iterrows()]

    # 创建 Geo 图表实例
    geo_chart = (
        Geo()
        .add_schema(
            maptype=city_name,
            itemstyle_opts=opts.ItemStyleOpts(color="#323c48", border_color="#111"),
        )
        # 添加热力图层
        .add(
            "充电站热力图",
            heatmap_data,
            type_=ChartType.HEATMAP,
        )
        # 添加已访问站点的散点图层
        .add(
            "已访问站点",
            visited_points,
            type_=ChartType.EFFECT_SCATTER,
            symbol_size=8,
            color='blue',
        )
        .set_series_opts(label_opts=opts.LabelOpts(is_show=False))
        .set_global_opts(
            visualmap_opts=opts.VisualMapOpts(
                is_show=True,
                min_=0,
                max_=1, # 因为我们的权重都是1，所以这里设为1
                range_color=['rgba(0,255,255,0.3)', 'rgba(0,255,0,0.6)', 'yellow', 'red']
            ),
            title_opts=opts.TitleOpts(
                title=f"{city_name} 充电站分布热力图及已访问站点",
                pos_left="center",
                pos_top="top",
                title_textstyle_opts=opts.TextStyleOpts(color="white")
            ),
            legend_opts=opts.LegendOpts(
                is_show=True,
                pos_left="left",
                orient="vertical",
                textstyle_opts=opts.TextStyleOpts(color="white")
            )
        )
    )
    
    # 注入高德地图 JS 文件
    geo_chart.js_dependencies.add("amap")
    geo_chart.add_js_funcs(f'echarts.registerMap("{city_name}", {{}}, {{}});')
    geo_chart.add_js_funcs(f'bmap.AMap.AMAP_KEY = "{amap_api_key}";')


    return geo_chart

print("✅ 热力图生成函数 'create_city_heatmap_with_visited_stations' 已准备就绪。")
print("请在下一个单元格中加载您的数据并调用此函数。")


In [ ]:
import pandas as pd
national_charging_data_file = 'national_charge_station.csv'
chargeing_station_df = pd.read_csv(national_charging_data_file)

In [ ]:

import folium
from folium.plugins import HeatMap

def create_city_heatmap_with_folium(all_stations_df, visited_stations_df):
    """
    使用 Folium 创建城市充电站热力图，并在图上标注已访问的站点。

    :param all_stations_df: 包含所有充电站的 DataFrame，需要有 'latitude', 'longitude' 列。
    :param visited_stations_df: 包含已访问充电站的 DataFrame，需要有 'latitude', 'longitude', 和 'station_name' 列。
    :return: folium.Map 对象
    """
    # --- 1. 设置地图中心和底图 ---
    # 计算地图中心点
    map_center = [all_stations_df['latitude'].mean(), all_stations_df['longitude'].mean()]
    
    # 定义高德底图
    gaode_tiles = "https://webrd01.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=8&x={x}&y={y}&z={z}"
    gaode_attribution = "Amap"

    # 创建 Folium 地图对象
    m = folium.Map(
        location=map_center,
        zoom_start=10,
        tiles=gaode_tiles,
        attr=gaode_attribution
    )

    # --- 2. 添加热力图层 ---
    # 准备热力图数据
    heat_data = [[row['latitude'], row['longitude']] for index, row in all_stations_df.iterrows()]
    HeatMap(heat_data).add_to(m)

    # --- 3. 添加已访问站点的标记 ---
    for idx, row in visited_stations_df.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=5,
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.7,
            popup=folium.Popup(row['station_name'], max_width=200) # 点击显示站点名称
        ).add_to(m)
        
    return m

print("✅ 使用 Folium 的热力图生成函数 'create_city_heatmap_with_folium' 已准备就绪。")
print("请在下一个单元格中加载您的数据并调用此函数。")


In [ ]:
city_ = "成都市"



all_stations_df = chargeing_station_df[chargeing_station_df["city"] == city_]
visited_stations_df = pd.read_csv("output/all_map_stations_CD.csv")

m = create_city_heatmap_with_folium(all_stations_df, visited_stations_df)
m.show_in_browser()

In [ ]:

import folium
from folium.plugins import HeatMap
import json

def create_city_heatmap_with_folium(all_stations_df, visited_stations_df, geojson_path=None):
    """
    使用 Folium 创建城市充电站热力图，标注已访问站点，并可选择性地绘制地理边界。

    :param all_stations_df: 包含所有充电站的 DataFrame，需要有 'latitude', 'longitude' 列。
    :param visited_stations_df: 包含已访问充电站的 DataFrame，需要有 'latitude', 'longitude', 和 'station_name' 列。
    :param geojson_path: (可选) GeoJSON 文件的路径，用于绘制地理边界。
    :return: folium.Map 对象
    """
    # --- 1. 设置地图中心和底图 ---
    map_center = [all_stations_df['latitude'].mean(), all_stations_df['longitude'].mean()]
    gaode_tiles = "https://webrd01.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=8&x={x}&y={y}&z={z}"
    gaode_attribution = "Amap"

    m = folium.Map(
        location=map_center,
        zoom_start=9, # 稍微缩小一点以便看到整个边界
        tiles=gaode_tiles,
        attr=gaode_attribution
    )

    # --- 2. (可选) 添加地理边界图层 ---
    if geojson_path:
        try:
            folium.GeoJson(
                geojson_path,
                style_function=lambda feature: {
                    'fillColor': 'none', # 边界内部不填充颜色
                    'color': 'red',      # 边界线颜色
                    'weight': 2,         # 边界线宽度
                    'dashArray': '5, 5'  # 虚线样式
                }
            ).add_to(m)
        except Exception as e:
            print(f"加载 GeoJSON 文件时出错: {e}")
            print("请检查文件路径是否正确，以及文件是否为有效的 GeoJSON 格式。")


    # --- 3. 添加热力图层 ---
    heat_data = [[row['latitude'], row['longitude']] for index, row in all_stations_df.iterrows()]
    HeatMap(heat_data).add_to(m)

    # --- 4. 添加已访问站点的标记 ---
    for idx, row in visited_stations_df.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=5,
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.7,
            popup=folium.Popup(row['station_name'], max_width=200)
        ).add_to(m)
        
    return m

print("✅ 更新后的热力图函数 'create_city_heatmap_with_folium' 已准备就绪，增加了对 GeoJSON 边界的支持。")
print("请在下一个单元格中加载数据，并在调用函数时传入 'geojson_path' 参数。")


In [ ]:

import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import HeatMap

# --- 1. 定义要处理的城市和相关信息 ---
cities_info = {
    '成都市': {
        'province_name': '成都市',
        'all_stations_file': 'national_charge_station.csv', # 假设都从这个总文件筛选
        'visited_stations_file': 'output/all_map_stations_CD.csv',
        'boundary_file': 'sichuan_boundary.geojson'
    },
    '重庆市': {
        'province_name': '重庆市',
        'all_stations_file': 'national_charge_station.csv',
        'visited_stations_file': 'output/all_map_stations_CQ.csv',
        'boundary_file': 'chongqing_boundary.geojson'
    },
    '贵阳市': {
        'province_name': '贵州省',
        'all_stations_file': 'national_charge_station.csv',
        'visited_stations_file': 'output/all_map_stations_GY.csv',
        'boundary_file': 'guizhou_boundary.geojson'
    }
}

shapefile_path = 'Province/province.shp'

# --- 2. 循环处理每个城市 ---
for city_name, info in cities_info.items():
    
    print(f"\\n{'='*20} 正在处理: {city_name} {'='*20}")
    
    # --- a. 准备地理边界文件 (如果不存在) ---
    if not pd.io.common.file_exists(info['boundary_file']):
        print(f"未找到边界文件 {info['boundary_file']}，正在从 Shapefile 创建...")
        try:
            gdf = gpd.read_file(shapefile_path, encoding='gbk')
            province_gdf = gdf[gdf['NAME'] == info['province_name']]
            if not province_gdf.empty:
                province_gdf.to_file(info['boundary_file'], driver='GeoJSON', encoding='utf-8')
                print(f"✅ 成功创建 {info['boundary_file']}")
            else:
                print(f"❌ 错误: 在 Shapefile 中找不到省份 '{info['province_name']}'")
                continue # 跳过这个城市
        except Exception as e:
            print(f"❌ 创建边界文件时出错: {e}")
            continue # 跳过这个城市
    else:
        print(f"已找到边界文件: {info['boundary_file']}")

    # --- b. 加载充电站数据 ---
    try:
        print("正在加载充电站数据...")
        # 从全国数据中筛选出当前城市的站点
        national_df = pd.read_csv(info['all_stations_file'])
        all_stations_df = national_df[national_df['city'] == city_name]
        
        # 加载已访问的站点
        visited_stations_df = pd.read_csv(info['visited_stations_file'])
        
        if all_stations_df.empty or visited_stations_df.empty:
            print(f"❌ 错误: '{city_name}' 的充电站数据为空，请检查文件名和数据内容。")
            continue
            
    except FileNotFoundError as e:
        print(f"❌ 错误: 找不到数据文件 {e.filename}。")
        continue
    except Exception as e:
        print(f"❌ 加载数据时出错: {e}")
        continue

    # --- c. 调用函数生成地图 ---
    print("正在生成地图...")
    city_map = create_city_heatmap_with_folium(
        all_stations_df=all_stations_df,
        visited_stations_df=visited_stations_df,
        geojson_path=info['boundary_file']
    )
    
    # --- d. 在 Notebook 中显示地图 ---
    print(f"--- {city_name} 地图预览 ---")
    display(city_map)

print(f"\\n{'='*50}")
print("所有城市处理完毕。")


In [ ]:

geo_df = gpd.read_file('District/district.shp', )

city_list = ["成都市", "重庆城区", "贵阳市"]
merged_geo_df = geo_df[geo_df["ct_name"].isin(city_list)]
# ["万州区", "黔江区", ]
merged_geo_df = merged_geo_df[~merged_geo_df["dt_name"].isin(["万州区", "黔江区", ])]

# cd_geo_df = geo_df[geo_df["ct_name"] == "成都市"]
# cq_geo_df = geo_df[geo_df["ct_name"] == "重庆城区"]
# gy_geo_df = geo_df[geo_df["ct_name"] == "贵阳市"]

In [ ]:
import pandas as pd
import folium

# --- 1. 筛选和合并地理数据 ---
try:
    city_list = ["成都市", "重庆城区", "贵阳市"]
    merged_geo_df = geo_df[geo_df["ct_name"].isin(city_list)]
    
    # 排除重庆市的特定区域
    merged_geo_df = merged_geo_df[~merged_geo_df["dt_name"].isin(["万州区", "黔江区", "开州区", "梁平区"])]
    print("✅ 成功筛选并合并了三个城市的地理数据。")

    # --- 2. 准备地图 ---
    gaode_tiles = "https://webrd01.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=8&x={x}&y={y}&z={z}"
    gaode_attribution = "Amap"

    # --- 3. 计算合并后的中心点并创建地图 ---
    # 必须转换为 EPSG:4326 (WGS84) 坐标系，folium 才能正确处理
    center_point = merged_geo_df.to_crs(epsg=4326).unary_union.centroid
    map_center = [center_point.y, center_point.x]
  
    m = folium.Map(
        location=map_center,
        zoom_start=7, # 调整缩放级别以容纳所有区域
        tiles=gaode_tiles,
        attr=gaode_attribution
    )

    # --- 4. 添加合并后的边界图层 ---
    # folium.GeoJson 会自动处理 GeoDataFrame
    folium.GeoJson(
        merged_geo_df,
        style_function=lambda feature: {
            'fillColor': 'blue',
            'color': 'blue',
            'weight': 2,
            'fillOpacity': 0.1,
        }
    ).add_to(m)

    # --- 5. 显示地图 ---
    print("--- 正在显示合并后的边界地图 ---")
    # 在新浏览器标签页中显示地图，效果更佳
    m.show_in_browser()
    # 如果您希望在 Notebook 中直接内联显示，请取消下面这行的注释
    # display(m)

except NameError as e:
    print(f"❌ 错误: 无法找到所需的 GeoDataFrame。请确保 {e} 变量已在之前的单元格中被正确定义。")
except Exception as e:
    print(f"❌ 处理地图时发生未知错误: {e}")

In [ ]:
national_df = pd.read_csv('national_charge_station.csv')
city_list = ["成都市", "重庆市", "贵阳市"]
city_stations_df = national_df[national_df["city"].isin(city_list)].reset_index(drop=True)
city_stations_df.head()

In [ ]:
# city_stations_df[city_stations_df["city"] == "重庆市"]


In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap

# --- 1. 筛选和合并地理数据 ---
try:
	city_list = ["成都市", "重庆城区", "贵阳市"]
	merged_geo_df = geo_df[geo_df["ct_name"].isin(city_list)]

	merged_geo_df = merged_geo_df[~merged_geo_df["dt_name"].isin(["万州区", "黔江区", "开州区", "梁平区"])]
	print("✅ 成功筛选并合并了三个城市的地理数据。")

	# --- 2. 准备地图 ---
	gaode_tiles = "https://webrd01.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=8&x={x}&y={y}&z={z}"
	gaode_attribution = "Amap"

	# --- 3. 计算合并后的中心点并创建地图 ---
	center_point = merged_geo_df.to_crs(epsg=4326).unary_union.centroid
	map_center = [center_point.y, center_point.x]

	m = folium.Map(
			location=map_center,
			zoom_start=7,
			tiles=gaode_tiles,
			attr=gaode_attribution
	)

	# --- 4. 添加合并后的边界图层 ---
	folium.GeoJson(
			merged_geo_df,
			style_function=lambda feature: {
					'fillColor': 'blue',
					'color': 'blue',
					'weight': 2,
					'fillOpacity': 0.1,
			}
	).add_to(m)
	print("✅ 成功添加地理边界图层。")

	# --- 5. 添加热力图层 ---
	# 使用您已创建的 city_stations_df
	# 准备热力图所需的数据格式 [[lat, lng], [lat, lng], ...]
	dt_list = merged_geo_df["dt_name"].tolist()
	city_stations_df = city_stations_df[city_stations_df["district"].isin(dt_list)]
	heat_data = [[row['latitude'], row['longitude']] for index, row in city_stations_df.iterrows()]

	# 添加热力图层，并设置较小的 radius 和 blur 以获得更细的颗粒度
	HeatMap(
			heat_data, 
			radius=8, 
			blur=5
	).add_to(m)
	print("✅ 成功添加热力图层（已调整颗粒度）。")

	# --- 6. 显示地图 ---
	print("--- 正在显示带有热力图的合并边界地图 ---")
	m.show_in_browser()
# display(m)

except NameError as e:
	print(f"❌ 错误: 无法找到所需的 DataFrame。请确保 {e} 变量已在之前的单元格中被正确定义。")
except Exception as e:
	print(f"❌ 处理地图时发生未知错误: {e}")


In [ ]:
cd_target_stations_df = pd.read_csv("output/all_map_stations_CD.csv")
cq_target_stations_df = pd.read_csv("output/all_map_stations_CQ.csv")
gy_target_stations_df = pd.read_csv("output/all_map_stations_GY.csv")

In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap

# --- 1. 筛选和合并地理数据 ---
try:
    city_list = ["成都市", "重庆城区", "贵阳市"]
    merged_geo_df = geo_df[geo_df["ct_name"].isin(city_list)]
    
    merged_geo_df = merged_geo_df[~merged_geo_df["dt_name"].isin(["万州区", "黔江区", "开州区", "梁平区"])]
    print("✅ 成功筛选并合并了三个城市的地理数据。")

    # --- 2. 准备地图 ---
    gaode_tiles = "https://webrd01.is.autonavi.com/appmaptile?lang=zh_cn&size=1&scale=1&style=8&x={x}&y={y}&z={z}"
    gaode_attribution = "Amap"

    # --- 3. 计算合并后的中心点并创建地图 ---
    center_point = merged_geo_df.to_crs(epsg=4326).unary_union.centroid
    map_center = [center_point.y, center_point.x]
  
    m = folium.Map(
        location=map_center,
        zoom_start=7,
        tiles=gaode_tiles,
        attr=gaode_attribution
    )

    # --- 4. 添加合并后的边界图层 ---
    folium.GeoJson(
        merged_geo_df,
        style_function=lambda feature: {
            'fillColor': 'blue',
            'color': 'blue',
            'weight': 2,
            'fillOpacity': 0.1,
        }
    ).add_to(m)
    print("✅ 成功添加地理边界图层。")

    # --- 5. 添加热力图层 ---
    heat_data = [[row['latitude'], row['longitude']] for index, row in city_stations_df.iterrows()]
    HeatMap(heat_data, radius=8, blur=5).add_to(m)
    print("✅ 成功添加热力图层。")

    # --- 6. 新增：加载并添加已访问站点的标记 ---
    print("正在加载并标记已访问的站点...")
    cd_target_stations_df = pd.read_csv("output/all_map_stations_CD.csv")
    cq_target_stations_df = pd.read_csv("output/all_map_stations_CQ.csv")
    gy_target_stations_df = pd.read_csv("output/all_map_stations_GY.csv")
    
    all_visited_stations_df = pd.concat([cd_target_stations_df, cq_target_stations_df, gy_target_stations_df], ignore_index=True)
    
    for idx, row in all_visited_stations_df.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=3, # 标记点的大小
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.7,
            popup=folium.Popup(row['station_name'], max_width=200) # 点击显示站点名称
        ).add_to(m)
    print(f"✅ 成功添加 {len(all_visited_stations_df)} 个已访问站点的标记。")

    # --- 7. 显示最终地图 ---
    print("--- 正在显示最终的合并地图 ---")
    m.show_in_browser()
    # display(m)

except NameError as e:
    print(f"❌ 错误: 无法找到所需的 DataFrame。请确保 {e} 变量已在之前的单元格中被正确定义。")
except Exception as e:
    print(f"❌ 处理地图时发生未知错误: {e}")

In [ ]:
import pandas as pd

# 读取并展示 Logbook_2026.xlsx 的内容
logbook_file = 'output/Logbook_2026.xlsx'
logbook_df = pd.read_excel(logbook_file)
logbook_df.head()

In [ ]:
import pandas as pd
import glob
import folium
from folium.plugins import HeatMap
import os

# --- 1. 设置 API Key 并加载任务记录 ---
AMAP_API_KEY = "e04966e153cf5a14405299a75cadafdd"
geocoded_cache_file = 'geocoded_mission_stations.csv'

# 检查是否已有缓存的地理编码文件
if os.path.exists(geocoded_cache_file):
    print(f"正在从缓存文件 '{geocoded_cache_file}' 加载已编码的站点坐标...")
    actual_visited_stations_df = pd.read_csv(geocoded_cache_file)
else:
    print("未找到缓存文件，将通过 API 进行地理编码。")
    # 使用之前单元格加载的 logbook_df
    try:
        # 提取不重复的场站名称
        unique_station_names = logbook_df['Station'].dropna().unique()
        print(f"从 Logbook.xlsx 中成功提取 {len(unique_station_names)} 个不重复的场站名称。")
    except NameError:
        raise NameError("'logbook_df' 未定义。请确保您已经运行了加载 'Logbook_2026.xlsx' 的单元格。")

    # --- 2. 调用 API 获取坐标 ---
    # 传入 station_names 列表和您的 API key
    actual_visited_stations_df = geocode_stations_amap(
        station_names=unique_station_names, 
        api_key=AMAP_API_KEY
    )
    # 删除任何未能成功编码的行
    actual_visited_stations_df.dropna(subset=['longitude', 'latitude'], inplace=True)
    # 保存结果以备将来使用
    actual_visited_stations_df.to_csv(geocoded_cache_file, index=False, encoding='utf-8-sig')
    print(f"已将编码结果保存到 '{geocoded_cache_file}'。")


# --- 3. 绘制最终地图 (边界 + 热力图 + 实际访问点) ---
try:
    print("\n--- 正在绘制最终地图 ---")
    # 复用之前的地图创建逻辑
    center_point = merged_geo_df.to_crs(epsg=4326).unary_union.centroid
    map_center = [center_point.y, center_point.x]
  
    final_map = folium.Map(
        location=map_center,
        zoom_start=7,
        tiles=gaode_tiles,
        attr=gaode_attribution
    )

    # 添加边界
    folium.GeoJson(merged_geo_df, style_function=lambda x: {'fillColor': 'blue', 'color': 'blue', 'weight': 2, 'fillOpacity': 0.1}).add_to(final_map)
    print("✅ 已添加地理边界。")

    # 添加热力图
    heat_data = [[row['latitude'], row['longitude']] for index, row in city_stations_df.iterrows()]
    HeatMap(heat_data, radius=8, blur=5).add_to(final_map)
    print("✅ 已添加热力图。")

    # 添加实际访问过的站点标记
    for idx, row in actual_visited_stations_df.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=3,
            color='red', # 使用红色以示区别
            fill=True,
            fill_color='red',
            fill_opacity=0.8,
            popup=folium.Popup(row['station_name'], max_width=200)
        ).add_to(final_map)
    print(f"✅ 成功添加 {len(actual_visited_stations_df)} 个实际访问站点的标记。")

    # --- 4. 显示地图 ---
    print("--- 正在显示最终的合并地图 ---")
    # final_map.show_in_browser()
    display(final_map)

except NameError as e:
    print(f"❌ 错误: 无法找到所需的 DataFrame。请确保 {e} 变量已在之前的单元格中被正确定义。")
except Exception as e:
    print(f"❌ 处理地图时发生未知错误: {e}")

In [ ]:
import requests
import pandas as pd
import time

def geocode_stations_amap(station_names, api_key, city=''):
    """
    使用高德地图地理编码 API (v3/geocode/geo) 将充电站名称列表转换为经纬度。

    :param station_names: 包含充电站名称的列表 (list of strings)。
    :param api_key: 您的高德 Web 服务 API 密钥。
    :param city: (可选) 指定城市名称，以提高搜索精度，例如 "成都市"。
    :return: 一个包含 'station_name', 'longitude', 'latitude' 列的 DataFrame。
    """
    geocoded_data = []
    print(f"准备开始地理编码 {len(station_names)} 个站点...")
    
    # 使用您指定的 geocode/geo 接口
    url = 'https://restapi.amap.com/v3/geocode/geo'

    for i, name in enumerate(station_names):
        params = {
            'key': api_key,
            'address': name, # 参数从 'keywords' 改为 'address'
            'city': city
        }
        
        try:
            response = requests.get(url, params=params)
            data = response.json()
            
            # 解析 geocode/geo 接口的返回结构
            if data['status'] == '1' and data['geocodes']:
                geocode = data['geocodes'][0]
                location = geocode['location']
                lng, lat = map(float, location.split(','))
                geocoded_data.append({'station_name': name, 'longitude': lng, 'latitude': lat})
                print(f"({i+1}/{len(station_names)}) ✅ 成功: {name} -> ({lng}, {lat})")
            else:
                geocoded_data.append({'station_name': name, 'longitude': None, 'latitude': None})
                print(f"({i+1}/{len(station_names)}) ❌ 失败: 未找到 '{name}' 的坐标。")

        except Exception as e:
            print(f"请求 API 时发生错误: {e}")
        
        # 加上一个小的延时，避免因请求过快而被 API 限制
        time.sleep(0.02)
        
    print("地理编码完成。")
    return pd.DataFrame(geocoded_data)

print("✅ 地理编码函数 'geocode_stations_amap' 已更新并准备就绪。")

In [ ]:
import pandas as pd
import glob
import folium
from folium.plugins import HeatMap
import os

# --- 1. 设置 API Key 并加载任务记录 ---
AMAP_API_KEY = "e04966e153cf5a14405299a75cadafdd"
geocoded_cache_file = 'geocoded_mission_stations.csv'

# 检查是否已有缓存的地理编码文件
if os.path.exists(geocoded_cache_file):
    print(f"正在从缓存文件 '{geocoded_cache_file}' 加载已编码的站点坐标...")
    actual_visited_stations_df = pd.read_csv(geocoded_cache_file)
else:
    print("未找到缓存文件，将通过 API 进行地理编码。")
    # 使用之前单元格加载的 logbook_df
    try:
        # 提取不重复的场站名称
        unique_station_names = logbook_df['Station Name'].dropna().unique()
        print(f"从 Logbook.xlsx 中成功提取 {len(unique_station_names)} 个不重复的场站名称。")
    except NameError:
        raise NameError("'logbook_df' 未定义。请确保您已经运行了加载 'Logbook_2026.xlsx' 的单元格。")

    # --- 2. 调用 API 获取坐标 ---
    # 传入 station_names 列表和您的 API key
    actual_visited_stations_df = geocode_stations_amap(
        station_names=unique_station_names, 
        api_key=AMAP_API_KEY
    )
    # 删除任何未能成功编码的行
    actual_visited_stations_df.dropna(subset=['longitude', 'latitude'], inplace=True)
    # 保存结果以备将来使用
    actual_visited_stations_df.to_csv(geocoded_cache_file, index=False, encoding='utf-8-sig')
    print(f"已将编码结果保存到 '{geocoded_cache_file}'。")


# --- 3. 绘制最终地图 (边界 + 热力图 + 品牌化标记) ---
try:
    print("\n--- 正在绘制最终地图 ---")
    # 复用之前的地图创建逻辑
    center_point = merged_geo_df.to_crs(epsg=4326).unary_union.centroid
    map_center = [center_point.y, center_point.x]
  
    final_map = folium.Map(
        location=map_center,
        zoom_start=7,
        tiles=gaode_tiles,
        attr=gaode_attribution
    )

    # 添加边界 (颜色改为灰色)
    folium.GeoJson(
        merged_geo_df, 
        style_function=lambda x: {'fillColor': '#808080', 'color': '#808080', 'weight': 2, 'fillOpacity': 0.1}
    ).add_to(final_map)
    print("✅ 已添加地理边界 (灰色)。")

    # 添加热力图
    heat_data = [[row['latitude'], row['longitude']] for index, row in city_stations_df.iterrows()]
    HeatMap(heat_data, radius=8, blur=5).add_to(final_map)
    print("✅ 已添加热力图。")

    # 添加实际访问过的站点标记 (大头针样式)
    for idx, row in actual_visited_stations_df.iterrows():
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=folium.Popup(row['station_name'], max_width=200),
            icon=folium.Icon(color='blue', icon='car', prefix='fa') # 使用蓝色汽车图标
        ).add_to(final_map)
    print(f"✅ 成功添加 {len(actual_visited_stations_df)} 个 BMW 风格的站点标记。")

    # --- 4. 显示地图 ---
    print("--- 正在显示最终的合并地图 ---")
    final_map.show_in_browser()

except NameError as e:
    print(f"❌ 错误: 无法找到所需的 DataFrame。请确保 {e} 变量已在之前的单元格中被正确定义。")
except Exception as e:
    print(f"❌ 处理地图时发生未知错误: {e}")

## CPO 統計

In [ ]:
!pip install pypinyin

In [ ]:
try:
    from pypinyin import pinyin, Style
except ImportError:
    print("错误：'pypinyin' 库未安装。请运行之前的单元格来安装它。")

def to_pinyin_flat(name):
    """将中文名转换为小写、以下划线分隔的无音调拼音字符串。"""
    # 检查输入是否为字符串，如果不是则返回 None
    if not isinstance(name, str):
        return None
    # 将中文转换为拼音列表，例如 [['xiǎo'], ['jú']]
    pinyin_list = pinyin(name, style=Style.NORMAL)
    # 将列表展平并用下划线连接，例如 "xiao_ju"
    return '_'.join(item[0] for item in pinyin_list)

try:
    # 检查 'logbook_df' 是否已定义
    if 'logbook_df' in locals() or 'logbook_df' in globals():
        print("正在为 'logbook_df' 的 'CPO Name' 列生成拼音...")
        # 将函数应用到 'CPO Name' 列，并将结果存储在新列 'cpo_name_pinyin' 中
        logbook_df['cpo_name_pinyin'] = logbook_df['CPO Name'].apply(to_pinyin_flat)
        
        print("已成功添加 'cpo_name_pinyin' 列。")
        
        # 显示包含新列的 DataFrame 的前几行以供检查
        display(logbook_df[['CPO Name', 'cpo_name_pinyin']].head())
    else:
        print("错误：DataFrame 'logbook_df' 未定义。请先运行加载 'Logbook_2026.xlsx' 的单元格。")

except Exception as e:
    print(f"处理过程中发生错误: {e}")
    
logbook_df.to_excel('output/Logbook_2026_with_pinyin.xlsx', index=False, )
print("已将包含拼音的新 Logbook 保存为 'output/Logbook_2026_with_pinyin.xlsx'。")		

In [ ]:
import pandas as pd

# --- 1. 定义城市简码映射 ---
city_code_map = {
    '成都': 'CD',
    '重庆': 'CQ',
    '贵阳': 'GY'
    # 如果有更多城市，可以继续在这里添加
}

try:
    # --- 2. 数据清洗与准备 ---
    # 【修正点】在处理前，先筛选掉关键信息（City, CPO Name, Station）为空的行
    required_cols = ['City', 'CPO Name', 'Station', 'Date']
    original_rows = len(logbook_df)
    logbook_df.dropna(subset=required_cols, inplace=True)
    cleaned_rows = len(logbook_df)
    if original_rows > cleaned_rows:
        print(f"已清理 {original_rows - cleaned_rows} 行关键信息为空的记录。")

    # 确保数据按时间排序
    logbook_df['datetime'] = pd.to_datetime(logbook_df['Date'])
    logbook_df.sort_values('datetime', inplace=True)
    print("已根据 'Date' 列完成时间排序。")

    # --- 3. 识别每个站点的首次出现时间 ---
    first_appearance_df = logbook_df.drop_duplicates(subset=['City', 'Station'], keep='first').copy()
    print(f"已识别出 {len(first_appearance_df)} 个不重复的充电站。")

    # --- 4. 为每个CPO内的站点分组并生成序号 ---
    first_appearance_df['cpo_station_rank'] = first_appearance_df.groupby(['City', 'cpo_name_pinyin']).cumcount() + 1
    
    # --- 5. 创建唯一的站点ID ---
    first_appearance_df['station_seq_code'] = first_appearance_df['cpo_station_rank'].apply(lambda x: f"{int(x):03d}")
    
    first_appearance_df['city_code'] = first_appearance_df['City'].map(city_code_map)
    
    # 【修正点】再次检查是否有因为城市不在映射表中而产生的NaN
    if first_appearance_df['city_code'].isnull().any():
        unmapped_cities = first_appearance_df[first_appearance_df['city_code'].isnull()]['City'].unique()
        print(f"⚠️ 警告: 以下城市在 city_code_map 中没有对应的简码，将无法生成ID: {unmapped_cities}")
        first_appearance_df.dropna(subset=['city_code'], inplace=True)

    first_appearance_df['station_unique_id'] = first_appearance_df['city_code'] + '_' + \
                                             first_appearance_df['cpo_name_pinyin'] + '_' + \
                                             first_appearance_df['station_seq_code']

    # --- 6. 将唯一ID合并回原始的logbook_df ---
    id_mapping_df = first_appearance_df[['City', 'Station', 'station_unique_id']]
    
    logbook_df = pd.merge(logbook_df, id_mapping_df, on=['City', 'Station'], how='left')
    
    print("已成功生成并添加 'station_unique_id' 列。")
    
    # --- 7. 显示结果以供检查 ---
    display(logbook_df[['datetime', 'City', 'CPO Name', 'Station', 'cpo_name_pinyin', 'station_unique_id']].head())

except KeyError as e:
    print(f"❌ 错误: DataFrame 中缺少必要的列: {e}。请确保 'logbook_df' 包含 'City', 'Station', 和 'cpo_name_pinyin'。")
except Exception as e:
    print(f"❌ 处理过程中发生未知错误: {e}")

In [ ]:
logbook_df.to_excel('output/Logbook_2026_with_station_id.xlsx', index=False, )
print("已将包含站点ID的新 Logbook 保存为 'output/Logbook_2026_with_station_id.xlsx'。")		

## 提取製造商

In [ ]:
import pandas as pd

df = pd.read_excel('log_book_final.xlsx')
df["Manufacturer"].unique()

In [ ]:
df2 = pd.read_excel('Logbook_2026.xlsx')
df2["Manufacturer"].unique()

In [ ]:
import numpy as np

output_df = pd.DataFrame({
		"Manufacturer": df2["Manufacturer"].unique(),
    # "canonical_name": np.nan, 
    # "pinyin_name": np.nan
})
output_df.dropna(inplace=True)
output_df

In [ ]:
output_df.to_excel('manufacturer_canonical_names.xlsx', index=False, )
output_df.to_csv('manufacturer_canonical_names.csv', index=False, )
print("已创建 'manufacturer_canonical_names.xlsx' 和 'manufacturer_canonical_names.csv' 文件，请在其中填写每个 Manufacturer 的规范化名称（canonical_name）。")

## LOGBOOK 簡單探索

In [ ]:
import pandas as pd

CD_FILE = "Logbook_2026.xlsx"

df = pd.read_excel(CD_FILE)
df.head()

In [ ]:
df["CPO Name"].unique(), df["CPO Name"].nunique()

In [ ]:
import pandas as pd

df_2025 = pd.read_excel("log_book_final.xlsx")
df_2026 = pd.read_excel("Logbook_2026.xlsx")

In [ ]:
print(df_2026.columns)
df_2026.head(3)

In [ ]:
# 以 2026 年的字段為標準，創建一個列名映射字典
column_mapping = {
    '天數': 'Day',
    '日期': 'Date',
    '站點': 'Station',
    '狀態': 'Status',
    '製造商': 'Manufacturer',
    'Rated Output Voltage(V)': 'Voltage(V)',
    'Rated Output Current(A)': 'Current(A)',
    'Rated Output Power(kW)': 'Power(kW)',
    '開啟電裝方式': 'Start Method',
    '12/24V power or Not': 'Has_12A_24A',
    '開始時間': 'Start Time',
    'Start SOC(%)': 'Start SoC(%)',
    '結束時間': 'End Time',
    'End SOC(%)': 'End SoC(%)',
    'Stop Method/Reason': 'End Method', # 注意：2025的“结束原因”对应2026的“End Method”
    '測試結果': 'Test Result',
    'Comment': 'Remark'
}

# 使用 rename 方法更新 df_2025 的列名
df_2025_renamed = df_2025.rename(columns=column_mapping)
# 為 2025 年的數據框添加 'City' 列，並賦值為 '广州'
df_2025_renamed['City'] = '广州'



# 顯示重命名後的列名，以確認操作成功
print("Renamed columns for 2025 data:")
print(df_2025_renamed.columns)

# Start SoC(%)
# Start SOC(%)


# 顯示前幾行以確認 'City' 列已成功添加
df_2025_renamed.head(3)

In [ ]:
cols = ['City', 'Day', 'Date', 'Station', 'Use Case', 'Status', 'CPO Name',
       'Manufacturer', 'MODEL', 'Voltage(V)', 'Current(A)', 'Power(kW)',
       'Start Method', 'Has_12A_24A', 'Start Time', 'Start SoC(%)', 'End Time',
       'End SoC(%)', 'End Method', 'Test Result', 'Remark']

df_2025_renamed = df_2025_renamed[cols]

In [ ]:
# 統一 'Status' 列的內容
status_replace_map = {
    'Normal': 'Normal Test',
    'Abnormal': 'Site Issue' # 將 Abnormal 映射為一個統一的場地問題標識
}
df_2025_renamed['Status'] = df_2025_renamed['Status'].replace(status_replace_map)

# 顯示 'Status' 列的值計數，以驗證替換是否成功
print("Value counts for 'Status' column in 2025 data after replacement:")
print(df_2025_renamed['Status'].value_counts())

In [ ]:
# 14. 根據報告分析，驗證 CRO (充電樁辨識報文) 相關信號

# 根據報告，根本原因是充電樁未能發送 CRO 報文。
# 我們來查找與 CRO 相關的信號，驗證這一點。

# 搜索與 CRO 相關的信號
cro_signals_list = [s for s in mdf_error.channels_db if 'cro' in s.lower()]

print(f"找到 {len(cro_signals_list)} 個 CRO 相關信號: {cro_signals_list}")

# 提取並繪製這些信號
if not cro_signals_list:
    print("在日誌文件中未找到任何與 CRO 相關的信號。")
else:
    try:
        df_cro = mdf_error.filter(cro_signals_list).to_dataframe()

        # 處理混合數據類型，只繪製數值類型
        numeric_cols = df_cro.select_dtypes(include='number').columns
        
        if not numeric_cols.empty:
            fig = make_subplots(rows=len(numeric_cols), cols=1, shared_xaxes=True, subplot_titles=numeric_cols)
            
            for i, col in enumerate(numeric_cols):
                # 使用階梯圖 (hv) 更能清晰地表示信號值的跳變
                fig.add_trace(go.Scatter(x=df_cro.index, y=df_cro[col], mode='lines', name=col, line_shape='hv'), row=i+1, col=1)

            fig.update_layout(height=200 * len(numeric_cols), title_text="CRO (充電樁辨識報文) 相關信號分析")
            fig.show()
        else:
            print("找到的 CRO 相關信號均為非數值類型，無法繪圖。")

        # 打印非數值類型的信號內容
        non_numeric_cols = df_cro.select_dtypes(exclude='number').columns
        if not non_numeric_cols.empty:
            print("\n--- 非數值 CRO 信號內容 ---")
            for col in non_numeric_cols:
                non_empty_values = df_cro[col].dropna().unique()
                if len(non_empty_values) > 0:
                    print(f"信號 '{col}':")
                    for val in non_empty_values:
                        if isinstance(val, bytes):
                            try:
                                print(f"  - {val.decode('utf-8', errors='ignore')}")
                            except:
                                print(f"  - {val}")
                        else:
                            print(f"  - {val}")
                else:
                    print(f"信號 '{col}' 沒有數據。")

    except Exception as e:
        print(f"提取或繪製 CRO 信號時出錯: {e}")

In [ ]:
# 15. 查找 CRM 和 CRO 的信號序列，驗證 CRO 的缺失

# 根據充電流程，車輛發送 CRM (車輛辨識報文) 後，應收到充電樁的 CRO (充電樁辨識報文)。
# 我們來驗證這個時序。

# 搜索 CRM 相關信號
crm_signals_list = [s for s in mdf_error.channels_db if 'crm' in s.lower()]
print(f"找到 {len(crm_signals_list)} 個 CRM 相關信號: {crm_signals_list}")

# 我們已經有了 cro_signals_list
print(f"找到 {len(cro_signals_list)} 個 CRO 相關信號: {cro_signals_list}")

# 提取 CRM 和 CRO 的數據
try:
    # 選擇幾個關鍵的、看起來不像內部狀態的信號進行分析
    # CRM: 通常是車輛發送給充電樁的，尋找類似 '...CrmVeh_rc...' (Vehicle) 的信號
    # CRO: 通常是充電樁發送的，尋找類似 '...CroChgr_rc...' (Charger) 的信號
    
    # 為了簡化，我們直接提取所有找到的信號
    all_sequence_signals = crm_signals_list + cro_signals_list
    df_sequence = mdf_error.filter(all_sequence_signals).to_dataframe()

    # 找出所有值發生變化的事件，並把它們放在一個列表裡
    events = []
    for col in df_sequence.columns:
        # 使用 dropna() 去掉 NaN 值，然後找出值變化的點
        changes = df_sequence[col].dropna().diff() != 0
        changed_indices = changes[changes].index
        
        for ts in changed_indices:
            events.append({
                'Timestamp': ts,
                'Signal': col,
                'Value': df_sequence.loc[ts, col]
            })

    # 按時間戳排序
    events.sort(key=lambda x: x['Timestamp'])

    print("\n--- CRM/CRO 事件序列 (按時間排序) ---")
    if not events:
        print("未找到任何 CRM 或 CRO 信號的變化事件。")
    else:
        for event in events:
            # 為了可讀性，對 Value 做一些格式化
            value_str = event['Value']
            if isinstance(value_str, bytes):
                value_str = value_str.decode('utf-8', errors='ignore')
            
            # 突出顯示關鍵信號
            if 'crm' in event['Signal'].lower():
                print(f"【車輛發送 CRM】 Timestamp: {event['Timestamp']}, Signal: {event['Signal']}, Value: {value_str}")
            elif 'cro' in event['Signal'].lower():
                print(f"【充電樁回應 CRO】 Timestamp: {event['Timestamp']}, Signal: {event['Signal']}, Value: {value_str}")
            else:
                print(f"    Timestamp: {event['Timestamp']}, Signal: {event['Signal']}, Value: {value_str}")

except Exception as e:
    print(f"提取或分析 CRM/CRO 序列時出錯: {e}")

In [ ]:
# 16. 打印解析後的信號變化序列

print("--- 已解析信號變化事件序列 (前 200 個事件) ---")
print("Timestamp (s)            Signal Name                                       Value")
print("------------------------------------------------------------------------------------------")

try:
    # 將所有信號轉換為 DataFrame
    all_signals_df = mdf_error.to_dataframe(raster=0.01) # 使用 raster 來對齊時間戳

    events = []
    
    # 記錄每個信號的最後一個值，以檢測變化
    last_values = {}

    # 遍歷 DataFrame 的每一行 (每一個時間點)
    for timestamp, row in all_signals_df.head(20000).iterrows(): # 限制分析的行數以提高效率
        for signal_name, value in row.items():
            # 檢查值是否有效 (非 NaN) 且與上一個值不同
            if pd.notna(value) and (signal_name not in last_values or not last_values[signal_name] == value):
                events.append({
                    'Timestamp': timestamp,
                    'Signal': signal_name,
                    'Value': value
                })
                last_values[signal_name] = value

    # 按時間戳排序 (雖然iterrows本身有序，但以防萬一)
    events.sort(key=lambda x: x['Timestamp'])

    # 打印前 200 個事件
    for event in events[:200]:
        value_str = event['Value']
        if isinstance(value_str, bytes):
            # 將字節串轉換為十六進制表示，更像原始報文
            value_str = value_str.hex().upper()
        
        print(f"{event['Timestamp']:<24.6f} {event['Signal']:<50} {value_str}")

except Exception as e:
    print(f"讀取信號序列時出錯: {e}")

In [ ]:

import can

logfile = "Day2/1129_14.36_GAC_UC4.blf"

with can.BLFReader(logfile) as reader:
    for msg in reader:
        print(
            f"Time: {msg.timestamp:.6f}, "
            f"ID: {hex(msg.arbitration_id)}, "
            f"DLC: {msg.dlc}, "
            f"Data: {msg.data.hex()}, "
            f"Channel: {msg.channel}"
        )


In [12]:
import can
import pandas as pd

# 指定您的 BLF 文件路径
logfile = "Day2/1129_13.30_Zeekr_UC6.blf"

# 创建一个空列表，用来存放所有报文数据
log_data = []

print(f"正在读取日志文件: {logfile} ...")

# 使用 with 语句安全地打开文件
with can.BLFReader(logfile) as reader:
    for msg in reader:
        # 将每条报文的信息作为一个字典追加到列表中
        log_data.append({
            'Timestamp': msg.timestamp,
            'ID': msg.arbitration_id,  # 直接存为整数，便于后续筛选
            'DLC': msg.dlc,
            'Data': msg.data.hex(),
            'Channel': msg.channel
        })

print("文件读取完毕，正在创建 DataFrame...")

# 从列表创建 Pandas DataFrame
df_blf = pd.DataFrame(log_data)

print("DataFrame 创建成功！")

# 为了方便查看，我们给 ID 列也增加一个十六进制的显示列
df_blf['ID_hex'] = df_blf['ID'].apply(hex)

# 显示 DataFrame 的前几行，确认结果
df_blf.head()

正在读取日志文件: Day2/1129_13.30_Zeekr_UC6.blf ...
文件读取完毕，正在创建 DataFrame...
DataFrame 创建成功！


,Timestamp,ID,DLC,Data,Channel,ID_hex
0,1.764422e+09,911,8,ebf9fc24fc24fcf4,1,0x38f
1,1.764422e+09,128,24,6cb458ff7fff7fff7fff7f111107f0ff12369af56f6ad69c,1,0x80
2,1.764422e+09,181,32,95c42b007de0fefb000ed007d007d007d007d007d0d7fc...,1,0xb5
3,1.764422e+09,182,48,4070db409c000040fe409c0000d0fec409e0ffffde3d00...,1,0xb6
4,1.764422e+09,213,16,ddf6000e3000ded0123252536873270a,1,0xd5


### 2. 分析正確的故障日誌：`1129_13.30_Zeekr_UC6.blf`
現在我們將分析流程應用於已知的故障文件，以驗證根本原因。

In [3]:
# 19. 加載 Zeekr 故障日誌並轉換為 DataFrame

import can
import pandas as pd

# 這次我們使用已知的故障日誌文件
faulty_logfile = "Day2/1129_13.30_Zeekr_UC6.blf"

log_data_faulty = []
print(f"正在讀取已知的故障日誌文件: {faulty_logfile} ...")

try:
    with can.BLFReader(faulty_logfile) as reader:
        for msg in reader:
            log_data_faulty.append({
                'Timestamp': msg.timestamp,
                'ID': msg.arbitration_id,
                'DLC': msg.dlc,
                'Data': msg.data.hex(),
                'Channel': msg.channel
            })
    
    print("故障日誌讀取完畢，正在創建 DataFrame...")
    df_blf_faulty = pd.DataFrame(log_data_faulty)
    df_blf_faulty['ID_hex'] = df_blf_faulty['ID'].apply(hex)
    
    print("故障日誌 DataFrame 創建成功！")
    display(df_blf_faulty.head())

except FileNotFoundError:
    print(f"錯誤：文件未找到！請確認路徑 '{faulty_logfile}' 是否正確。")
except Exception as e:
    print(f"處理過程中發生錯誤: {e}")

正在讀取已知的故障日誌文件: Day2/1129_13.30_Zeekr_UC6.blf ...
故障日誌讀取完畢，正在創建 DataFrame...
故障日誌 DataFrame 創建成功！


,Timestamp,ID,DLC,Data,Channel,ID_hex
0,1.764422e+09,911,8,ebf9fc24fc24fcf4,1,0x38f
1,1.764422e+09,128,24,6cb458ff7fff7fff7fff7f111107f0ff12369af56f6ad69c,1,0x80
2,1.764422e+09,181,32,95c42b007de0fefb000ed007d007d007d007d007d0d7fc...,1,0xb5
3,1.764422e+09,182,48,4070db409c000040fe409c0000d0fec409e0ffffde3d00...,1,0xb6
4,1.764422e+09,213,16,ddf6000e3000ded0123252536873270a,1,0xd5


In [5]:
reader

In [2]:
# 20. 在 Zeekr 故障日誌中篩選 CRM 和 CRO 報文

# 現在對新創建的、正確的故障 DataFrame (df_blf_faulty) 執行相同的篩選邏輯

print(f"在故障日誌中篩選關鍵報文...")
print(f"車輛 (CRM) ID: {hex(CRM_ID)}")
print(f"充電樁 (CRO) ID: {hex(CRO_ID)}")

df_handshake_faulty = df_blf_faulty[df_blf_faulty['ID'].isin([CRM_ID, CRO_ID])].copy()

df_handshake_faulty['MessageType'] = df_handshake_faulty['ID'].apply(label_message)

# 顯示結果
if df_handshake_faulty.empty:
    print("\n【結論】: 在 Zeekr 故障日誌中，完全沒有找到車輛(CRM)或充電樁(CRO)的握手報文。")
    print("這極不尋常，可能意味著日誌記錄的範圍沒有覆蓋到插槍握手階段。")
else:
    print("\n分析結果：已從 Zeekr 故障日誌中篩選出握手階段的相關報文。")
    
    message_types_found = df_handshake_faulty['MessageType'].unique()
    
    if 'Vehicle -> CRM' in message_types_found and 'Charger -> CRO' not in message_types_found:
        print("\n【最終結論】: 找到了車輛發送的 CRM 報文，但完全沒有找到充電樁回應的 CRO 報文。")
        print("這從底層數據證實了 'Root Reason: Charge failed to send CRO' 的判斷是完全正確的。")
    elif 'Charger -> CRO' in message_types_found and 'Vehicle -> CRM' not in message_types_found:
        print("\n【最終結論】: 找到了充電樁發送的 CRO 報文，但沒有找到車輛發送的 CRM 報文。")
        print("這非常罕見，可能表示車輛通信模塊有問題或日誌記錄不完整。")
    elif 'Vehicle -> CRM' in message_types_found and 'Charger -> CRO' in message_types_found:
        print("\n【最終結論】: 車輛和充電樁的握手報文都存在。通信看起來是建立的，問題可能出在後續階段。")
    
    display(df_handshake_faulty)

在故障日誌中篩選關鍵報文...


NameError: name 'CRM_ID' is not defined

In [72]:
# 21. 验证 BRO, CRO, CST 的时序关系

# 定义国标充电过程中的关键报文 ID
BRO_ID = 0x1806F456  # BMS Ready to charge OK (车辆BMS准备就绪)
CRO_ID = 0x182756F4  # Charger Recognition (充电桩辨识)
CST_ID = 0x1813F456  # Charger Stop Transaction (充电桩中止充电)
CRM_ID = 0x1826F456  # Vehicle Recognition (车辆辨识)

print("开始筛选 BRO, CRO, CST, CRM 报文...")

# 将需要分析的 ID 放入一个列表
key_sequence_ids = [BRO_ID, CRO_ID, CST_ID, CRM_ID]

# 从 Zeekr 故障日志 DataFrame 中筛选
df_sequence_analysis = df_blf_faulty[df_blf_faulty['ID'].isin(key_sequence_ids)].copy()

# 定义一个更详细的报文类型标签函数
def label_sequence_message(can_id):
    if can_id == CRM_ID:
        return 'Vehicle -> CRM (车辆辨识)'
    elif can_id == CRO_ID:
        return 'Charger -> CRO (充电桩辨识)'
    elif can_id == BRO_ID:
        return 'Vehicle -> BRO (BMS准备就绪)'
    elif can_id == CST_ID:
        return 'Charger -> CST (中止充电)'
    return 'Other'

# 应用标签
df_sequence_analysis['MessageType'] = df_sequence_analysis['ID'].apply(label_sequence_message)

# 检查结果并给出分析
if df_sequence_analysis.empty:
    print("\n【分析结论】: 在日志中未找到任何 BRO, CRO, CST 或 CRM 报文。")
else:
    print("\n【分析结论】: 已找到相关报文，请查看下方的时序表。")
    
    # 检查是否存在 BRO -> CST 的序列
    bro_events = df_sequence_analysis[df_sequence_analysis['ID'] == BRO_ID]
    cst_events = df_sequence_analysis[df_sequence_analysis['ID'] == CST_ID]
    
    if not bro_events.empty and not cst_events.empty:
        first_bro_time = bro_events['Timestamp'].min()
        first_cst_time = cst_events['Timestamp'].min()
        
        if first_bro_time < first_cst_time:
            print(f"\n关键发现：检测到车辆在 {first_bro_time:.6f} 发送了 BRO (准备就绪)，")
            print(f"随后充电桩在 {first_cst_time:.6f} 发送了 CST (中止充电)。")
            print("这符合 'BRO -> CST' 的故障模式。")
            
            # 检查 BRO 和 CST 之间是否有 CRO
            cro_between = df_sequence_analysis[
                (df_sequence_analysis['ID'] == CRO_ID) &
                (df_sequence_analysis['Timestamp'] > first_bro_time) &
                (df_sequence_analysis['Timestamp'] < first_cst_time)
            ]
            if not cro_between.empty:
                print("并且，在 BRO 和 CST 之间，我们确实找到了 CRO 报文。")
                print("这说明握手成功了，但问题出在握手之后的某个环节，最终导致了充电中止。")
            else:
                print("并且，在 BRO 和 CST 之间，我们【没有】找到 CRO 报文。")
                print("这与 'BRO 和 CST 之间应该要有 CRO' 的规则相悖，强烈暗示问题与 CRO 报文的缺失或无效有关。")

# 按照时间戳排序并显示结果
display(df_sequence_analysis.sort_values(by='Timestamp'))

开始筛选 BRO, CRO, CST, CRM 报文...

【分析结论】: 已找到相关报文，请查看下方的时序表。


,Timestamp,ID,DLC,Data,Channel,ID_hex,MessageType
252139,1.764423e+09,405206102,3,010100,0,0x1826f456,Vehicle -> CRM (车辆辨识)
252820,1.764423e+09,405206102,3,010100,0,0x1826f456,Vehicle -> CRM (车辆辨识)
253526,1.764423e+09,405206102,3,010100,0,0x1826f456,Vehicle -> CRM (车辆辨识)
253974,1.764423e+09,405231348,2,4411,0,0x182756f4,Charger -> CRO (充电桩辨识)
254238,1.764423e+09,405206102,3,010100,0,0x1826f456,Vehicle -> CRM (车辆辨识)
254689,1.764423e+09,405231348,2,4411,0,0x182756f4,Charger -> CRO (充电桩辨识)
254946,1.764423e+09,405206102,3,010100,0,0x1826f456,Vehicle -> CRM (车辆辨识)
255378,1.764423e+09,405231348,2,4411,0,0x182756f4,Charger -> CRO (充电桩辨识)
255628,1.764423e+09,405206102,3,010100,0,0x1826f456,Vehicle -> CRM (车辆辨识)
256071,1.764423e+09,405231348,2,4411,0,0x182756f4,Charger -> CRO (充电桩辨识)


405231348

In [77]:
# 22. 在 MF4 文件中分析充电参数协商阶段 (BCP, BCL, BCS, CTS, CCS)

from asammdf import MDF
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 重新加载 MF4 故障文件
mdf_path = "Day2/11.29_zeekr_UC6_13.25_first_try_fialed.mf4"
mdf_error = MDF(mdf_path)

print(f"已重新加载 MF4 文件: {mdf_path}")

# 定义充电参数协商阶段的关键字
param_keywords = ['bcp', 'bcl', 'bcs', 'cts', 'ccs', 'bro', 'cst']

# 搜索包含这些关键字的信号
negotiation_signals = []
for keyword in param_keywords:
    # 避免重复添加
    found_signals = [s for s in mdf_error.channels_db if keyword in s.lower()]
    for sig in found_signals:
        if sig not in negotiation_signals:
            negotiation_signals.append(sig)

print(f"\n找到 {len(negotiation_signals)} 个与参数协商相关的信号:")
for sig in negotiation_signals:
    print(f" - {sig}")

# 提取并绘图
if not negotiation_signals:
    print("\n未找到任何与充电参数协商相关的信号。")
else:
    try:
        df_negotiation = mdf_error.filter(negotiation_signals).to_dataframe()

        # 只绘制数值类型的信号
        numeric_cols = df_negotiation.select_dtypes(include='number').columns
        
        # if not numeric_cols.empty:
        #     print("\n正在绘制数值型信号图表...")
        #     # 创建子图，每个信号一个
        #     fig = make_subplots(
        #         rows=len(numeric_cols), 
        #         cols=1, 
        #         shared_xaxes=True, 
        #         subplot_titles=[col.split('.')[-1] for col in numeric_cols] # 使用信号名作为标题
        #     )
            
        #     for i, col in enumerate(numeric_cols):
        #         # 使用阶梯图 (hv) 更能清晰地表示状态信号的跳变
        #         fig.add_trace(
        #             go.Scatter(x=df_negotiation.index, y=df_negotiation[col], mode='lines', name=col, line_shape='hv'),
        #             row=i + 1,
        #             col=1
        #         )

        #     fig.update_layout(
        #         height=150 * len(numeric_cols), 
        #         title_text="充电参数协商阶段信号分析 (MF4)",
        #         showlegend=False # 信号太多，图例意义不大
        #     )
        #     fig.show()
        # else:
        #     print("\n找到的信号均为非数值类型，无法绘图。")

        # 打印非数值类型的信号内容
        non_numeric_cols = df_negotiation.select_dtypes(exclude='number').columns
        if not non_numeric_cols.empty:
            print("\n--- 非数值信号内容 ---")
            for col in non_numeric_cols:
                non_empty_values = df_negotiation[col].dropna().unique()
                if len(non_empty_values) > 0:
                    print(f"信号 '{col}':")
                    for val in non_empty_values:
                        if isinstance(val, bytes):
                            print(f"  - {val.decode('utf-8', errors='ignore')}")
                        else:
                            print(f"  - {val}")

    except Exception as e:
        print(f"\n提取或绘制信号时出错: {e}")


已重新加载 MF4 文件: Day2/11.29_zeekr_UC6_13.25_first_try_fialed.mf4

找到 390 个与参数协商相关的信号:
 - BMW_SWC_CifOut.SenderPorts.ppBMW_rc_ChaInfoDcChnBcpVeh_rc.BMW_rc_ChaInfoDcChnBcpVeh_rc.BMW_u_MaxAllwdChaVoltCell
 - XCP:1.BMW_rc_ChaInfoDcChnBcpVeh_rc.BMW_u_MaxAllwdChaVoltCell
 - BMW_rc_ChaInfoDcChnBcpVeh_rc.BMW_u_MaxAllwdChaVoltCell
 - BMW_SWC_CifOut.SenderPorts.ppBMW_rc_ChaInfoDcChnBcpVeh_rc.BMW_rc_ChaInfoDcChnBcpVeh_rc.BMW_pct_SocBcp
 - XCP:1.BMW_rc_ChaInfoDcChnBcpVeh_rc.BMW_pct_SocBcp
 - BMW_rc_ChaInfoDcChnBcpVeh_rc.BMW_pct_SocBcp
 - BMW_SWC_IntBtdxCore1.SenderPorts.rpBMW_st_CemMsgToutDetdBcp_rcppBMW_st_CemMsgToutDetdBcp_rc.BMW_st_CemMsgToutDetdBcp_rc.BMW_st_CemMsgToutDetdBcp_qi
 - XCP:1.BMW_st_CemMsgToutDetdBcp_rc.BMW_st_CemMsgToutDetdBcp_qi
 - BMW_st_CemMsgToutDetdBcp_rc.BMW_st_CemMsgToutDetdBcp_qi
 - BMW_SWC_CifOut.SenderPorts.ppBMW_rc_ChaInfoDcChnBcpVeh_rc.BMW_rc_ChaInfoDcChnBcpVeh_rc.BMW_u_PretTotVoltHvs
 - XCP:1.BMW_rc_ChaInfoDcChnBcpVeh_rc.BMW_u_PretTotVoltHvs
 - BMW_rc_ChaInfoDcChnBcpVeh_

In [78]:
# 23. 可视化 BCP 超时事件的关键时序

import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("正在聚焦并可视化 BCP 超时事件的关键时序...")

# 锁定三个核心信号
key_timing_signals = [
    'BMW_st_BroChaRdyBms_rc.BMW_st_BroChaRdyBms',                # 1. 故事起点: 车辆准备就绪 (BRO)
    'BMW_st_CemMsgToutDetdBcp_rc.BMW_st_CemMsgToutDetdBcp_qi', # 2. 核心事件: BCP 超时检测信号
    'BMW_st_CstReasChaStopChgr_rc.BMW_st_CstReasChaStopChgr'   # 3. 故事结局: 充电桩中止原因 (CST)
]

try:
    # 提取这三个信号的数据
    df_timing = mdf_error.filter(key_timing_signals).to_dataframe()

    # 创建 3 个子图
    fig = make_subplots(
        rows=3, 
        cols=1, 
        shared_xaxes=True, 
        subplot_titles=("1. 车辆准备就绪 (BRO)", "2. BCP 超时检测 (Timeout)", "3. 充电桩中止 (CST)")
    )

    # 依次添加每个信号的轨迹
    # 使用阶梯图 (hv) 来清晰表示状态跳变
    fig.add_trace(go.Scatter(x=df_timing.index, y=df_timing[key_timing_signals[0]], mode='lines', name='BRO', line_shape='hv'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_timing.index, y=df_timing[key_timing_signals[1]], mode='lines', name='BCP Timeout', line_shape='hv'), row=2, col=1)
    fig.add_trace(go.Scatter(x=df_timing.index, y=df_timing[key_timing_signals[2]], mode='lines', name='CST Reason', line_shape='hv'), row=3, col=1)

    # 更新图表布局
    fig.update_layout(
        height=600, 
        title_text="故障诊断时序图：从车辆就绪到充电中止的全过程",
        showlegend=False
    )
    
    # 更新 Y 轴的标签，让图表更易读
    fig.update_yaxes(title_text="BRO Status", row=1, col=1)
    fig.update_yaxes(title_text="Timeout Status", row=2, col=1)
    fig.update_yaxes(title_text="CST Reason Code", row=3, col=1)

    fig.show()

except Exception as e:
    print(f"\n创建时序图时出错: {e}")


正在聚焦并可视化 BCP 超时事件的关键时序...

创建时序图时出错: Channel "BMW_st_BroChaRdyBms_rc.BMW_st_BroChaRdyBms" not found


In [83]:
# 24. 修正错误：精确查找信号名称

import asammdf
import os
import json

# 修正文件路径
correct_mf4_path = 'Day2/11.29_zeekr_UC6_13.25_first_try_fialed.mf4'
print(f"正在使用文件: {correct_mf4_path}")

mdf = asammdf.MDF(correct_mf4_path)

# 定义我们要查找的关键字
# 基于之前的分析，'Cem' 是一个很关键的前缀
search_keywords = ['BRO', 'BCP', 'CST', 'Timeout', 'ChaRdy', 'Stop', 'Cem']

# 搜索并打印所有信号名称
all_signals = mdf.channels_db

found_signals = {keyword: [] for keyword in search_keywords}

for signal_name in all_signals:
    for keyword in search_keywords:
        if keyword.lower() in signal_name.lower():
            found_signals[keyword].append(signal_name)

# 打印找到的信号
print("\n根据关键字找到的信号:")
print(json.dumps(found_signals, indent=2))

mdf.close()


正在使用文件: Day2/11.29_zeekr_UC6_13.25_first_try_fialed.mf4

根据关键字找到的信号:
{
  "BRO": [
    "BMW_SWC_CifOut.SenderPorts.ppBMW_rc_ChaInfoDcChnBroVeh_rc.BMW_rc_ChaInfoDcChnBroVeh_rc.BMW_rc_ChaInfoDcChnBroVeh_qi",
    "XCP:1.BMW_rc_ChaInfoDcChnBroVeh_rc.BMW_rc_ChaInfoDcChnBroVeh_qi",
    "BMW_rc_ChaInfoDcChnBroVeh_rc.BMW_rc_ChaInfoDcChnBroVeh_qi",
    "BMW_SWC_CifOut.SenderPorts.ppBMW_rc_ChaInfoDcChnBroVeh_rc.BMW_rc_ChaInfoDcChnBroVeh_rc.BMW_st_RdyChaVeh",
    "XCP:1.BMW_rc_ChaInfoDcChnBroVeh_rc.BMW_st_RdyChaVeh",
    "BMW_rc_ChaInfoDcChnBroVeh_rc.BMW_st_RdyChaVeh",
    "BMW_SWC_IntBtdxCore1.SenderPorts.rpBMW_st_CemMsgToutDetdBro_rcppBMW_st_CemMsgToutDetdBro_rc.BMW_st_CemMsgToutDetdBro_rc.BMW_st_CemMsgToutDetdBro_qi",
    "XCP:1.BMW_st_CemMsgToutDetdBro_rc.BMW_st_CemMsgToutDetdBro_qi",
    "BMW_st_CemMsgToutDetdBro_rc.BMW_st_CemMsgToutDetdBro_qi",
    "BMW_SWC_IntBtdxCore1.SenderPorts.rpBMW_st_CemMsgToutDetdBro_rcppBMW_st_CemMsgToutDetdBro_rc.BMW_st_CemMsgToutDetdBro_rc.BMW_st_CemMsgToutDetdBro

## 合併日誌

In [10]:
import pandas as pd	

logbook_file_1 = "G70_I460_Logbook_2025.xlsx"
logbook_file_2 = "G70_I490_Logbook_2026.xlsx"

logbook_df_1 = pd.read_excel(logbook_file_1)
logbook_df_2 = pd.read_excel(logbook_file_2)
print(f"已加载日志文件: {logbook_file_1} 和 {logbook_file_2}")

logbook_df_1["Vehicle_Model"] = "G70ILC"
logbook_df_2["Vehicle_Model"] = "G70ILC"

logbook_df_1["SW_Model"] = "I460"
logbook_df_2["SW_Model"] = "I490"

G70_ILC_df = pd.concat([logbook_df_1, logbook_df_2], ignore_index=True)

col_list = ['City', 'Vehicle_Model', 'SW_Model', 'Day', 'Date', 'Station', 
            'Use Case', 'Status', 'CPO Name', 'Manufacturer', 'MODEL', 
            'Voltage(V)', 'Current(A)', 'Power(kW)', 'Start Method', 
            'Has_12A_24A', 'Start Time', 'Start SoC(%)', 'End Time',
            'End SoC(%)', 'End Method', 'Test Result', 'Remark', 
            ]

G70_ILC_df = G70_ILC_df[col_list]

G70_ILC_df.to_excel("G70_ILC_Logbook.xlsx", index=False)
print("已成功合并日志文件并保存为 'G70_ILC_Logbook.xlsx'。")

已加载日志文件: G70_I460_Logbook_2025.xlsx 和 G70_I490_Logbook_2026.xlsx
已成功合并日志文件并保存为 'G70_ILC_Logbook.xlsx'。


In [11]:
G70_ILC_df

,City,Vehicle_Model,SW_Model,Day,Date,Station,Use Case,Status,CPO Name,Manufacturer,...,Power(kW),Start Method,Has_12A_24A,Start Time,Start SoC(%),End Time,End SoC(%),End Method,Test Result,Remark
0,广州,G70ILC,I460,1.0,11/28/25,广州城投（花城广场南区）充电站,DC_UC4_17460724,Normal Test,GZ Chengtou/城投充电,许记/Xuji,...,90.0,掃描 QRcode,NaN,11/28/25 11:19,36.0,11/28/25 11:24,37.0,APP,Pass,在內部發現空位
1,广州,G70ILC,I460,1.0,11/28/25,小鹏汽车充电站(小鹏超充广州国际金融中心IFC站),DC_UC5_17460720,Normal Test,CAMS,CAMS,...,NaN,掃描 QRcode,NaN,11/28/25 11:49,36.0,11/28/25 11:54,40.0,APP,Pass,同一区域内发现开麦斯
2,广州,G70ILC,I460,1.0,11/28/25,小鹏汽车充电站(小鹏超充广州国际金融中心IFC站),DC_UC4_17460724,Normal Test,Xpeng,East/易事特集团股份有限公司,...,250.0,掃描 QRcode,NaN,11/28/25 12:07,40.0,11/28/25 12:10,42.0,APP,Pass,该 USE CASE 测试时固定三分钟
3,广州,G70ILC,I460,1.0,11/28/25,IONCHI(小鹏超充广州国际金融中心IFC站),DC_UC5_17460720,Normal Test,IONCHI,英飞源/INFY,...,600.0,掃描 QRcode,NaN,11/28/25 12:19,42.0,11/28/25 12:23,50.0,APP,Pass,同一地點發現逸安啟 設置目標充電至 50%
4,广州,G70ILC,I460,1.0,11/28/25,小朗充电-广州市妇女儿童医疗中心充电站,DC_UC5_17460720,Site Issue,小朗/Xiaolang,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,該地區不適合測試 以後去掉 POI 醫院位置
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,贵阳,G70ILC,I490,1.0,2026-03-18 00:00:00,中国石化超级充电站(綦江赶水停车区超充站重庆捷充),UC01_CC_DC_AC_Sessions,Normal Test,倢電充,貴州鴻蒙,...,360.0,Scan QR Code,0.0,2026-03-18 14:44:33,16.0,2026-03-18 14:46:53,21.0,APP,Pass,NaN
163,贵阳,G70ILC,I490,1.0,2026-03-18 00:00:00,贵州高速超级充电站(桐梓服务区海口方向充电站·华为超充技术支持),UC01_CC_DC_AC_Sessions,Normal Test,倢電充,貴州鴻蒙,...,360.0,Scan QR Code,0.0,2026-03-18 14:44:33,16.0,2026-03-18 14:46:53,21.0,APP,Pass,NaN
164,贵阳,G70ILC,I490,1.0,2026-03-18 00:00:00,聚合快充腾龙湾汽车充电站,UC01_CC_DC_AC_Sessions,Normal Test,聚合快充,深圳聚合快充,...,NaN,Scan QR Code,0.0,2026-03-18 17:59:31,29.0,2026-03-18 17:59:35,32.0,LAT,Pass,NaN
165,贵阳,G70ILC,I490,1.0,2026-03-18 00:00:00,户户充汽车充电站(贵阳壹号新能源充电站),UC01_CC_DC_AC_Sessions,Normal Test,戶戶充,NaN,...,7.0,Scan QR Code,0.0,2026-03-18 18:10:31,24.0,2026-03-18 18:20:31,24.0,LAT,Pass,NaN


### 🚚 Step X: 處理與統一场站 ID (Station ID)
因為廣州 (GZ) 等站點過去命名不統一，且可能缺乏元數據 (meta.json)。
這裡我們抓取所有的獨特站點，重新賦予標準化的 `城市_CPO拼音_遞增序號`。
之後可以再把這個 mapping 表 merge 回主日誌。

In [12]:
import pandas as pd
from collections import defaultdict

# 假設 df 是合併好的 Logbooks。這裡我們使用示範資料。
# TODO: 請換成你實際合併完的 df
data = {
    'City': ['成都', '成都', '廣州', '廣州'],
    'CPO': ['特來電', '星星充電', '南方電網', '特來電'],
    'Station Name': ['成都特來電1號', '星星成都站', '南網廣州站', '廣州特來電二號']
}
df_demo = pd.DataFrame(data)

# 1. 城市拼音/縮寫對照表
city_map = {'成都': 'CD', '重慶': 'CQ', '貴陽': 'GY', '廣州': 'GZ', '北京': 'BJ'}

# 2. CPO 拼音對照表 (如果之前有 cpo_brand_aliases_with_pinyin.csv 可以讀進來轉 dict)
cpo_pinyin_map = {
    '特來電': 'te_lai_dian',
    '星星充電': 'xing_xing',
    '南方電網': 'nan_wang',
    '液冷': 'ye_leng' 
}

# 3. 抽取獨一無二的站點
unique_stations = df_demo[['City', 'CPO', 'Station Name']].dropna().drop_duplicates().reset_index(drop=True)

counters = defaultdict(int)

# 4. 生成標準化 Station_ID 的函式
def generate_station_id(row):
    city_code = city_map.get(row['City'], 'UN') # 找不到就 UN (Unknown)
    cpo_pinyin = cpo_pinyin_map.get(row['CPO'], 'unknown_cpo')
    
    prefix = f"{city_code}_{cpo_pinyin}"
    counters[prefix] += 1
    
    # 例如 CD_te_lai_dian_001
    return f"{prefix}_{counters[prefix]:03d}"

unique_stations['Station_ID'] = unique_stations.apply(generate_station_id, axis=1)

print("✅ 生成的標準化 Station Mapping:")
unique_stations.head()

# 下一步：把這個 unique_stations 用 pd.merge (how='left', on=['City', 'CPO', 'Station Name']) 接回去主表。

✅ 生成的標準化 Station Mapping:


,City,CPO,Station Name,Station_ID
0,成都,特來電,成都特來電1號,CD_te_lai_dian_001
1,成都,星星充電,星星成都站,CD_xing_xing_001
2,廣州,南方電網,南網廣州站,GZ_nan_wang_001
3,廣州,特來電,廣州特來電二號,GZ_te_lai_dian_001


In [13]:
import pandas as pd
from collections import defaultdict

# 1. 讀取 CPO 拼音對照表，建立 Mapping
try:
    df_cpo = pd.read_csv('cpo_brand_aliases_with_pinyin.csv')
    # 第一行可能是重複的標頭 (brand_name, cpo_name)，可以直接過濾或直接轉 dict
    cpo_pinyin_map = dict(zip(df_cpo['alias'].str.strip(), df_cpo['pinyin_name'].str.strip()))
except Exception as e:
    print(f"⚠️ 讀取 CPO 字典失敗: {e}")
    cpo_pinyin_map = {}

# 2. 城市拼音/縮寫對照表 (依照您專案常用的縮寫)
city_map = {
    '成都': 'CD', 
    '重庆': 'CQ', '重慶': 'CQ',
    '贵阳': 'GY', '貴陽': 'GY', 
    '广州': 'GZ', '廣州': 'GZ',
    '北京': 'BJ'
}

def create_station_master_list(df_merged):
    """
    傳入合併後的 df (df_merged)，此函式會抽取出獨一無二的站點，並賦予標準化的 Station_ID。
    要求 df_merged 至少包含 'City', 'CPO', 'Station Name' 這三個欄位。
    """
    # 抽取獨一無二的站點
    unique_stations = df_merged[['City', 'CPO', 'Station Name']].dropna().drop_duplicates().reset_index(drop=True)
    
    counters = defaultdict(int)
    
    def generate_station_id(row):
        city_raw = str(row['City']).strip()
        cpo_raw = str(row['CPO']).strip()
        
        # 轉換 City Code (預設 UN)
        city_code = city_map.get(city_raw, 'UN')
        
        # 轉換 CPO 拼音 (預設 unknown_cpo)
        cpo_pinyin = cpo_pinyin_map.get(cpo_raw, 'unknown_cpo')
        if cpo_pinyin == 'unknown_cpo':
            # 如果發現不在字典裡的，可以先印出來，方便實習生後續去補
            print(f"⚠️ 發現未登錄的 CPO 品牌: {cpo_raw}")
            
        prefix = f"{city_code}_{cpo_pinyin}"
        counters[prefix] += 1
        
        # 例如 CD_te_lai_dian_001
        return f"{prefix}_{counters[prefix]:03d}"

    # 套用生成 ID
    unique_stations['Station_ID'] = unique_stations.apply(generate_station_id, axis=1)
    
    return unique_stations

# --- 示範用法 👇 ---
# 假設 df_merged 是你剛剛合併所有 Excel 得到的 Dataframe
# df_station_master = create_station_master_list(df_merged)
# display(df_station_master)
# 
# 接著再用 merge 拼印回去原表：
# df_final = pd.merge(df_merged, df_station_master, on=['City', 'CPO', 'Station Name'], how='left')

In [16]:
# 自動讀取並合併我們的日誌檔案，直接看效果！
import os

files_to_merge = ['G70_I460_Logbook_2025.xlsx', 'G70_I490_Logbook_2026.xlsx']
df_list = []

for file in files_to_merge:
    if os.path.exists(file):
        temp_df = pd.read_excel(file)
        df_list.append(temp_df)
    else:
        print(f"⚠️ 找不到檔案：{file}")

if df_list:
    merged_df = pd.concat(df_list, ignore_index=True)
    
    # 確保欄位名稱正確。有些日誌的站點名稱可能叫 'Station' 或 'Station Name'
    if 'Station' in merged_df.columns and 'Station Name' not in merged_df.columns:
        merged_df.rename(columns={'Station': 'Station Name'}, inplace=True)
    if 'CPO Name' in merged_df.columns and 'CPO' not in merged_df.columns:
        merged_df.rename(columns={'CPO Name': 'CPO'}, inplace=True)

    print(f"📦 成功合併日誌，總筆數：{len(merged_df)}")
    
    # 使用我們先前寫好的函數生成 Station Master List
    df_station_master = create_station_master_list(merged_df)
    
    print("✅ 成功生成站點 Master List (節錄前 5 筆):")
    display(df_station_master.head())
    
    # 檢查看看有哪些 ID 被賦予了
    print(f"\n🏷️ 總共生成了 {len(df_station_master)} 個獨一無二的 Station_ID\n")
    
    # 把它合併回原始大表
    final_merged_df = pd.merge(merged_df, df_station_master, on=['City', 'CPO', 'Station Name'], how='left')
    
    print("🎯 最終成果 (合併了 Station_ID 的測試日誌):")
    cols_to_show = ['Date', 'City', 'CPO', 'Station Name', 'Station_ID', 'Status']
    # 確保這些欄位都存在再顯示
    cols_to_show = [col for col in cols_to_show if col in final_merged_df.columns]
    display(final_merged_df[cols_to_show].head(10))
    
    # 存檔備用
    final_merged_df.to_csv('master_logbook_enriched.csv', index=False, encoding='utf-8-sig')
    print("\n💾 已成功將成果儲存為: master_logbook_enriched.csv")
else:
    print("沒有找到可用的 Excel 日誌檔案。")

📦 成功合併日誌，總筆數：167
⚠️ 發現未登錄的 CPO 品牌: GZ Chengtou/城投充电
⚠️ 發現未登錄的 CPO 品牌: CAMS
⚠️ 發現未登錄的 CPO 品牌: Xpeng
⚠️ 發現未登錄的 CPO 品牌: IONCHI
⚠️ 發現未登錄的 CPO 品牌: 小朗/Xiaolang
⚠️ 發現未登錄的 CPO 品牌: XDT/GAC
⚠️ 發現未登錄的 CPO 品牌: NIO
⚠️ 發現未登錄的 CPO 品牌: 維傑斯/Vajesta
⚠️ 發現未登錄的 CPO 品牌: 維傑斯/Vajesta
⚠️ 發現未登錄的 CPO 品牌: 好充/Godd charging
⚠️ 發現未登錄的 CPO 品牌: 英非
⚠️ 發現未登錄的 CPO 品牌: 快電
⚠️ 發現未登錄的 CPO 品牌: BP
⚠️ 發現未登錄的 CPO 品牌: NIO
⚠️ 發現未登錄的 CPO 品牌: 新创能源
⚠️ 發現未登錄的 CPO 品牌: AINO
⚠️ 發現未登錄的 CPO 品牌: 順義衝
⚠️ 發現未登錄的 CPO 品牌: ZEEKR
⚠️ 發現未登錄的 CPO 品牌: Li Auto
⚠️ 發現未登錄的 CPO 品牌: GAC
⚠️ 發現未登錄的 CPO 品牌: EV Power
⚠️ 發現未登錄的 CPO 品牌: 周门充电站
⚠️ 發現未登錄的 CPO 品牌: 石化易電
⚠️ 發現未登錄的 CPO 品牌: IONCHI
⚠️ 發現未登錄的 CPO 品牌: 快來衝汽車充電站
⚠️ 發現未登錄的 CPO 品牌: 萬城萬沖
⚠️ 發現未登錄的 CPO 品牌: 鵬輝充電
⚠️ 發現未登錄的 CPO 品牌: South Gird
⚠️ 發現未登錄的 CPO 品牌: Teld
⚠️ 發現未登錄的 CPO 品牌: SC
⚠️ 發現未登錄的 CPO 品牌: 節電通
⚠️ 發現未登錄的 CPO 品牌: EAST DCWB
⚠️ 發現未登錄的 CPO 品牌: Xiaoju
⚠️ 發現未登錄的 CPO 品牌: Teld
⚠️ 發現未登錄的 CPO 品牌: Lvnenghuichong
⚠️ 發現未登錄的 CPO 品牌: Ruisuzhineng
⚠️ 發現未登錄的 CPO 品牌: zhongnengkunchng
⚠️ 發現未登錄的 CPO 品牌: teld
⚠️ 發現未登錄的 CPO

,City,CPO,Station Name,Station_ID
0,广州,GZ Chengtou/城投充电,广州城投（花城广场南区）充电站,GZ_unknown_cpo_001
1,广州,CAMS,小鹏汽车充电站(小鹏超充广州国际金融中心IFC站),GZ_unknown_cpo_002
2,广州,Xpeng,小鹏汽车充电站(小鹏超充广州国际金融中心IFC站),GZ_unknown_cpo_003
3,广州,IONCHI,IONCHI(小鹏超充广州国际金融中心IFC站),GZ_unknown_cpo_004
4,广州,小朗/Xiaolang,小朗充电-广州市妇女儿童医疗中心充电站,GZ_unknown_cpo_005



🏷️ 總共生成了 144 個獨一無二的 Station_ID

🎯 最終成果 (合併了 Station_ID 的測試日誌):


,Date,City,CPO,Station Name,Station_ID,Status
0,11/28/25,广州,GZ Chengtou/城投充电,广州城投（花城广场南区）充电站,GZ_unknown_cpo_001,Normal Test
1,11/28/25,广州,CAMS,小鹏汽车充电站(小鹏超充广州国际金融中心IFC站),GZ_unknown_cpo_002,Normal Test
2,11/28/25,广州,Xpeng,小鹏汽车充电站(小鹏超充广州国际金融中心IFC站),GZ_unknown_cpo_003,Normal Test
3,11/28/25,广州,IONCHI,IONCHI(小鹏超充广州国际金融中心IFC站),GZ_unknown_cpo_004,Normal Test
4,11/28/25,广州,小朗/Xiaolang,小朗充电-广州市妇女儿童医疗中心充电站,GZ_unknown_cpo_005,Site Issue
5,11/28/25,广州,XDT/GAC,广汽能源汽车充电站(广汽昊铂广州珠江公园超充站),GZ_unknown_cpo_006,Normal Test
6,11/28/25,广州,NIO,蔚来能源目的地充电站(内部使用广州天銮广场),GZ_unknown_cpo_007,Site Issue
7,11/28/25,广州,維傑斯/Vajesta,维杰斯通充电站-慢充,GZ_unknown_cpo_008,Normal Test
8,11/28/25,广州,維傑斯/Vajesta,维杰斯通充电站,GZ_unknown_cpo_009,Normal Test
9,11/28/25,广州,好充/Godd charging,好充穗盐西充电站,GZ_unknown_cpo_010,Normal Test



💾 已成功將成果儲存為: master_logbook_enriched.csv


### 🔎 Step 2: 整理與關聯 Trace 報文檔案 (.mf4 / .blf)
因為大家在現場存檔的檔名非常隨性（例如 `11.28_AION_UC6_11.42.mf4` 或 `1128_11.03_UC4_Touchengcharging.blf`）。
我們在這裡寫一個爬蟲，先去 `Day1` 跟 `Day2` 資料夾把所有 Trace 掃出來，並嘗試用語法（正則表達式）「猜出」它是哪一天的哪個 Use Case。

In [19]:
import os
import re
import pandas as pd

trace_folders = ['Day1', 'Day2']
trace_records = []

# 掃描資料夾
for folder in trace_folders:
    if os.path.exists(folder):
        for file in os.listdir(folder):
            if file.endswith(('.mf4', '.blf')): # 只抓報文
                
                # --- 核心邏輯：用正則表達式(Regex)去硬挖出藏在檔名裡的線索 ---
                
                # 1. 抓取 Use Case (UCX)
                uc_match = re.search(r'(UC\d+)', file, re.IGNORECASE)
                use_case = uc_match.group(1).upper() if uc_match else "Unknown"
                
                # 2. 抓取日期 (例如 11.28, 1128, 11.29)
                date_match = re.search(r'(11\.?2[89])', file)
                date_str = date_match.group(1).replace('.', '') if date_match else "Unknown" # 統一拔掉點，變成 1128 或 1129
                
                # 3. 抓取時間 (例如 14.48, 10.35, 13.18) 注意: 不要跟11.28日期撞車
                # 這裡的邏輯是: 找兩位數.兩位數，且開頭數字>08 (通常測試時間在白天) 
                time_match = re.search(r'([0-2]\d[.:][0-5]\d)', file)
                time_str = time_match.group(1).replace('.', ':') if time_match else "Unknown" # 轉成時分標準 14:48
                
                # 如果時間抓錯抓到日期(如11.28)，我們可以強制定個規則修復，例如如果time跟date一樣就清空
                if time_str.replace(':', '') == date_str:
                    time_str = "Conflict_With_Date"
                
                trace_records.append({
                    'Folder': folder,
                    'Filename': file,
                    'Extracted_Date': date_str,
                    'Extracted_Time': time_str,
                    'Extracted_UseCase': use_case
                })

df_traces = pd.DataFrame(trace_records)

print(f"📡 總共從資料夾中掃描到 {len(df_traces)} 個 Trace 檔案！")
print("快來看看我們從混亂的檔名中「榨出」了什麼線索：")
display(df_traces.head(15))

# 這樣一來，我們就有一張 Trace 的表了。
# 接下來我們可以透過 Extracted_Date 跟 Extracted_UseCase（甚至部分時間）去跟上面的 final_merged_df 進行「相親配對」！

📡 總共從資料夾中掃描到 47 個 Trace 檔案！
快來看看我們從混亂的檔名中「榨出」了什麼線索：


,Folder,Filename,Extracted_Date,Extracted_Time,Extracted_UseCase
0,Day1,11.28_Xpeng_UC4_12.03.mf4,1128,Conflict_With_Date,UC4
1,Day1,1128_17.09_UC4_Haochong_second 240kw.blf,1128,17:09,UC4
2,Day1,11.28_Gac_UC6_13.18.mf4,1128,Conflict_With_Date,UC6
3,Day1,1128_12.03_UC6_Xpeng.blf,1128,12:03,UC6
4,Day1,1128_11.46_UC5_CAMS.blf,1128,11:46,UC5
5,Day1,11.28_IONCHI_UC5_12.16.mf4,1128,Conflict_With_Date,UC5
6,Day1,1128_12.16_UC5_IONCHI.blf,1128,12:16,UC5
7,Day1,1128_11.03_UC4_Touchengcharging.blf,1128,11:03,UC4
8,Day1,1128_14.34_UC6_Weijiesi.blf,1128,14:34,UC6
9,Day1,1128_17.02_UC4_Haochong.blf,1128,17:02,UC4


In [20]:
df_traces

,Folder,Filename,Extracted_Date,Extracted_Time,Extracted_UseCase
0,Day1,11.28_Xpeng_UC4_12.03.mf4,1128,Conflict_With_Date,UC4
1,Day1,1128_17.09_UC4_Haochong_second 240kw.blf,1128,17:09,UC4
2,Day1,11.28_Gac_UC6_13.18.mf4,1128,Conflict_With_Date,UC6
3,Day1,1128_12.03_UC6_Xpeng.blf,1128,12:03,UC6
4,Day1,1128_11.46_UC5_CAMS.blf,1128,11:46,UC5
5,Day1,11.28_IONCHI_UC5_12.16.mf4,1128,Conflict_With_Date,UC5
6,Day1,1128_12.16_UC5_IONCHI.blf,1128,12:16,UC5
7,Day1,1128_11.03_UC4_Touchengcharging.blf,1128,11:03,UC4
8,Day1,1128_14.34_UC6_Weijiesi.blf,1128,14:34,UC6
9,Day1,1128_17.02_UC4_Haochong.blf,1128,17:02,UC4


In [21]:
import os
import re
import pandas as pd

trace_folders = ['Day1', 'Day2']
trace_records = []

# 掃描資料夾
for folder in trace_folders:
    if os.path.exists(folder):
        for file in os.listdir(folder):
            if file.endswith(('.mf4', '.blf')): # 只抓報文
                
                # --- 核心邏輯：用正則表達式(Regex)去硬挖出藏在檔名裡的線索 ---
                
                # 1. 抓取 Use Case (UCX)
                uc_match = re.search(r'(UC\d+)', file, re.IGNORECASE)
                use_case = uc_match.group(1).upper() if uc_match else "Unknown"
                
                # 2. 抓取日期 (例如 11.28, 1128, 11.29, 1129)
                date_match = re.search(r'(11\.?2[89])', file)
                date_str = date_match.group(1).replace('.', '') if date_match else "Unknown" # 統一拔掉點，變成 1128 或 1129
                
                # 3. 抓取時間 (例如 14.48, 10.35, 13.18) 
                time_match = re.search(r'([0-2]\d[.:][0-5]\d)', file)
                time_str = time_match.group(1).replace('.', ':') if time_match else "Unknown" # 轉成時分標準 14:48
                
                # 防呆: 如果時間抓到日期 (如 11:28)，通常那是日期，清空或忽略
                if time_str.replace(':', '') == date_str:
                    time_str = "Conflict_With_Date"
                
                trace_records.append({
                    'Folder': folder,
                    'Filename': file,
                    'Extracted_Date': date_str,
                    'Extracted_Time': time_str,
                    'Extracted_UseCase': use_case
                })

df_traces = pd.DataFrame(trace_records)

print(f"📡 總共從資料夾中掃描到 {len(df_traces)} 個 Trace 檔案！")
if len(df_traces) > 0:
    print("來看一下我們從混亂的檔名中「榨出」的特徵 (節錄前 15 筆)：")
    display(df_traces.head(15))
    
    # 隨便檢查幾筆時間萃取有問題的
    conflict_df = df_traces[df_traces['Extracted_Time'] == 'Conflict_With_Date']
    if not conflict_df.empty:
        print("\n⚠️ 以下幾筆的時間區段跟日期撞車了，需要後續特別照顧：")
        display(conflict_df)
else:
    print("⚠️ 奇怪，資料夾裡沒有找到 .mf4 或 .blf，請檢查副檔名與路徑。")

📡 總共從資料夾中掃描到 47 個 Trace 檔案！
來看一下我們從混亂的檔名中「榨出」的特徵 (節錄前 15 筆)：


,Folder,Filename,Extracted_Date,Extracted_Time,Extracted_UseCase
0,Day1,11.28_Xpeng_UC4_12.03.mf4,1128,Conflict_With_Date,UC4
1,Day1,1128_17.09_UC4_Haochong_second 240kw.blf,1128,17:09,UC4
2,Day1,11.28_Gac_UC6_13.18.mf4,1128,Conflict_With_Date,UC6
3,Day1,1128_12.03_UC6_Xpeng.blf,1128,12:03,UC6
4,Day1,1128_11.46_UC5_CAMS.blf,1128,11:46,UC5
5,Day1,11.28_IONCHI_UC5_12.16.mf4,1128,Conflict_With_Date,UC5
6,Day1,1128_12.16_UC5_IONCHI.blf,1128,12:16,UC5
7,Day1,1128_11.03_UC4_Touchengcharging.blf,1128,11:03,UC4
8,Day1,1128_14.34_UC6_Weijiesi.blf,1128,14:34,UC6
9,Day1,1128_17.02_UC4_Haochong.blf,1128,17:02,UC4



⚠️ 以下幾筆的時間區段跟日期撞車了，需要後續特別照顧：


,Folder,Filename,Extracted_Date,Extracted_Time,Extracted_UseCase
0,Day1,11.28_Xpeng_UC4_12.03.mf4,1128,Conflict_With_Date,UC4
2,Day1,11.28_Gac_UC6_13.18.mf4,1128,Conflict_With_Date,UC6
5,Day1,11.28_IONCHI_UC5_12.16.mf4,1128,Conflict_With_Date,UC5
12,Day1,11.28_haochong240_UC4_17.15.mf4,1128,Conflict_With_Date,UC4
13,Day1,11.28_Cams_UC5.mf4,1128,Conflict_With_Date,UC5
15,Day1,11.28_Chengtou_UC4.mf4,1128,Conflict_With_Date,UC4
16,Day1,11.28_weijiesi_UC6_14.43.mf4,1128,Conflict_With_Date,UC6
17,Day2,11.29_pingyunlu no2 charger_UC4_10.11.mf4,1129,Conflict_With_Date,UC4
18,Day2,11.29_Nio_UC5_11.03_second try.mf4,1129,Conflict_With_Date,UC5
19,Day2,11.29_Xinchuang_UC5_11.27.mf4,1129,Conflict_With_Date,UC5


### 🎯 Trace 報文檔案「黃金命名規格」(Golden Standard Naming Convention)

為了讓後續的系統（包含 Dashboard 分析與自動關聯）能 100% 準確抓到檔案，且不需要寫複雜的容錯程式碼，建議手動修改 Trace 檔名時遵循以下**絕對標準格式**：

> **格式：`[日期]_[Station_ID]_[UseCase]_[時間].[副檔名]`**

**📌 詳細欄位說明：**
1. **日期 (Date)**：4 碼數字，例如 `1128` 或 `1129`。
2. **Station_ID (場站唯一編號)**：請對照剛剛生成的 Master Data，例如 `GZ_xpeng_001` 或 `CD_te_lai_dian_002`。（**這超級重要！** 這樣 Trace 檔案和照片資料夾就能透過同一個 ID 直接打通！）
3. **場景 (Use Case)**：大寫 UC 加上數字，例如 `UC4`, `UC5`, `UC6`。
4. **時間 (Time)**：4 碼數字 (HHMM)，例如早上十點半為 `1030`，下午兩點為 `1400`。用來區分同一天在同一個站點跑了兩次以上的測試（例如 second try）。

**✅ 標準命名範例：**
* 原本：`11.28_Xpeng_UC4_12.03.mf4` ➔ **改成：`1128_GZ_xpeng_001_UC4_1203.mf4`**
* 原本：`1129_10.58_UC5_NIO_second try.blf` ➔ **改成：`1129_GZ_nio_002_UC5_1058.blf`**
* 原本：`11.29_Nio_UC5_11.03_second try.mf4` ➔ **改成：`1129_GZ_nio_002_UC5_1103.mf4`**

*(註：您可以把剛剛匯出的 `master_logbook_enriched.csv` 發給實習生，讓他們對照著 Excel 上的 `Station_ID` 和 `Start Time` 去逐一把資料夾裡的 Trace 檔案重新命名。)*

In [ ]:
mf4_file = "Day1/11.28_Cams_UC5.mf4"



### 📈 Step 3: 從報文 (.mf4 / .blf) 萃取「充電曲線」輕量化檔案
要把幾百 MB 的 Trace 報文直接放進 Dashboard 即時讀取會非常卡頓。最聰明的做法是：**寫個腳本把 Trace 裡面的核心物理量（電壓、電流、功率、SoC）抽出來，並且降頻（例如每 1 秒採樣一次），存成輕巧的 `_curve.csv`。**

到時候，Dashboard (Tab 3) 點擊某筆紀錄時，只要去讀這個幾 KB 的 Curve CSV 就可以秒繪製充電折線圖了！

In [ ]:
import os
from asammdf import MDF

def extract_charging_curve_from_mf4(mf4_path, output_dir="output_curves"):
    """
    從 .mf4 檔案提取指定的充電信號 (電壓、電流、功率、SoC)
    並降頻取樣 (Downsample) 到 1Hz (每秒1筆)，存成極小體積的 CSV 或 Parquet。
    """
    try:
        # 1. 讀取 MDF 報文
        mdf = MDF(mf4_path)
        
        print(f"正在分析: {mf4_path}...")
        
        # 2. 定義你需要找的物理訊號 (這需要跟負責寫 DLT (Data Logger) / CAN DBC 的同事拿確切的變數名)
        # TODO: 請換成真實的變數名稱！(例如: BMS_Volt_HV, OBC_Curr_Out, SOC_Disp...)
        target_signals = [
            "BMS_Voltage_HV_Signal_Name",    # 車端高壓電壓
            "BMS_Current_HV_Signal_Name",    # 車端高壓電流
            "Charger_Power_Output_Name",     # 樁端輸出功率
            "BMS_SOC_Display"                # 儀表 SoC
        ]
        
        # 檢查該檔案裡有哪幾個你想找的訊號
        available_signals = [sig for sig in target_signals if sig in mdf.channels_db]
        
        if not available_signals:
            print(f"⚠️ 在 {mf4_path} 找不到指定的信號！(可能是變數名稱不對哦)")
            return None
        
        # 3. 把原始 10ms (100Hz) 的高頻資料，降採樣到 1秒1筆 (Raster=1)，減少檔案體積 (節省 99% 的容量！)
        df_curve = mdf.to_dataframe(channels=available_signals, raster=1)
        
        # 4. 存檔到輕量化的資料夾
        os.makedirs(output_dir, exist_ok=True)
        base_name = os.path.basename(mf4_path).replace('.mf4', '_curve.csv')
        out_path = os.path.join(output_dir, base_name)
        
        # 存成輕量化的 CSV
        df_curve.to_csv(out_path)
        print(f"✅ 成功萃取輕量化充電曲線並存入: {out_path}")
        
        return out_path

    except Exception as e:
        print(f"❌ 讀取 {mf4_path} 失敗: {e}")
        return None

# 👇 您可以拿某個昨天測試成功的 .mf4 檔案名來試跑看看
# test_file = "Day2/11.29_zeekr_UC6_13.37_second_try_succeed.mf4" (記得換成剛剛規範改好的新檔名)
# extract_charging_curve_from_mf4(test_file)

In [22]:
from asammdf import MDF

mf4_file = "Day1/11.28_Cams_UC5.mf4"
print(f"🔍 正在探索檔案: {mf4_file}")

try:
    mdf = MDF(mf4_file)
    
    # 為了找出實際的變數名，我們先印出包含特定關鍵字的 channel (電壓、電流、SOC、功率)
    keywords = ['volt', 'curr', 'soc', 'power', 'bms', 'cpo']
    found_channels = {k: [] for k in keywords}
    
    for signal in mdf.channels_db:
        for k in keywords:
            if k.lower() in signal.lower():
                found_channels[k].append(signal)
                
    for k in keywords:
        print(f"👉 包含 '{k}' 的信號有 {len(found_channels[k])} 個, 節錄前 5 個:")
        print(found_channels[k][:5])
        print("-")
        
    mdf.close()
except FileNotFoundError:
    print(f"⚠️ 找不到檔案 {mf4_file}，請確認路徑。")
except Exception as e:
    print(f"⚠️ 解析發生錯誤: {e}")

🔍 正在探索檔案: Day1/11.28_Cams_UC5.mf4
👉 包含 'volt' 的信號有 75 個, 節錄前 5 個:
['BMW_SWC_eTmcEin.SenderPorts.ppBMWetmc_u_HvsHtrVoltAct.BMWetmc_u_HvsHtrVoltAct', 'XCP:1.BMWetmc_u_HvsHtrVoltAct', 'BMWetmc_u_HvsHtrVoltAct', 'BMW_SWC_HvStr.SenderPorts.ppBMWhvstr_b_HvSysLoVoltThd_bo.BMWhvstr_b_HvSysLoVoltThd_bo', 'XCP:1.BMWhvstr_b_HvSysLoVoltThd_bo']
-
👉 包含 'curr' 的信號有 0 個, 節錄前 5 個:
[]
-
👉 包含 'soc' 的信號有 24 個, 節錄前 5 個:
['BMW_SWC_CifIn.SenderPorts.ppBMWcif_soc_HvsDisp_si.BMWcif_soc_HvsDisp_si', 'XCP:1.BMWcif_soc_HvsDisp_si', 'BMWcif_soc_HvsDisp_si', 'BMW_SWC_HvChaPre.SenderPorts.ppBMWhvcha_soc_HvbAct_uw.BMWhvcha_soc_HvbAct_uw', 'XCP:1.BMWhvcha_soc_HvbAct_uw']
-
👉 包含 'power' 的信號有 0 個, 節錄前 5 個:
[]
-
👉 包含 'bms' 的信號有 0 個, 節錄前 5 個:
[]
-
👉 包含 'cpo' 的信號有 0 個, 節錄前 5 個:
[]
-


In [23]:
# 看來這些信號命名有它獨特的德系風格 (BMW...)
# 再下深一點的關鍵字去找出 電流 (Current, Ampere, Curr, Cur) 和 功率 (Power, Pwr, Watt)
try:
    mdf = MDF(mf4_file)
    
    # 擴充關鍵字
    keywords = ['amp', 'curr', 'cur', 'pwr', 'pow', 'max', 'req', 'act']
    found_channels = {k: [] for k in keywords}
    
    for signal in mdf.channels_db:
        # 單獨針對電流跟功率相關的模糊搜索
        if 'cur' in signal.lower():
            found_channels['cur'].append(signal)
        if 'pwr' in signal.lower():
            found_channels['pwr'].append(signal)
            
    print(f"👉 包含 'cur' (Current) 相關信號 {len(found_channels['cur'])} 個，節錄前 10 個:")
    print(found_channels['cur'][:10])
    
    print(f"\n👉 包含 'pwr' (Power) 相關信號 {len(found_channels['pwr'])} 個，節錄前 10 個:")
    print(found_channels['pwr'][:10])
    
    mdf.close()
except Exception as e:
    pass

👉 包含 'cur' (Current) 相關信號 0 個，節錄前 10 個:
[]

👉 包含 'pwr' (Power) 相關信號 238 個，節錄前 10 個:
['BMWhvcha_pwr_AcDispPwrCha_sw_KW', 'BMWhvcha_pwr_DcDispPwrCha_si_KW', 'BMW_SWC_HvChaPre.PerInstanceMemories.BMWhvcha_pwr_ChaDcdcPrdn_sw', 'XCP:1.BMWhvcha_pwr_ChaDcdcPrdn_sw', 'BMWhvcha_pwr_ChaDcdcPrdn_sw', 'BMW_SWC_HvChaPre.SenderPorts.ppBMWhvcha_pwr_DispPwrChaInf_uw.BMWhvcha_pwr_DispPwrChaInf_uw', 'XCP:1.BMWhvcha_pwr_DispPwrChaInf_uw', 'BMWhvcha_pwr_DispPwrChaInf_uw', 'BMW_SWC_HvChaSigOut.SenderPorts.ppBMW_pwr_DispPwrCha_rc.BMW_pwr_DispPwrCha_rc.BMW_pwr_DispPwrCha', 'XCP:1.BMW_pwr_DispPwrCha_rc.BMW_pwr_DispPwrCha']


In [24]:
# 如果您記得確切的訊號名稱（例如電壓是 'BMS_Volt_xxx', 或是 'BMWhvcha_...' 開頭）
# 這裡有一個快速腳本直接試跑一次降採樣匯出！

def extract_specific_signals(mf4_file, signal_names):
    from asammdf import MDF
    import os
    
    try:
        mdf = MDF(mf4_file)
        # 過濾存在於此檔案的信號
        valid_signals = [sig for sig in signal_names if sig in mdf.channels_db]
        
        if not valid_signals:
            print("❌ 找不到您指定的任何信號！")
            return
        
        print(f"📦 找到 {len(valid_signals)} 個信號，開始降採樣(Raster=1s)...")
        # raster=1 代表每秒一筆
        df_curve = mdf.to_dataframe(channels=valid_signals, raster=1)
        
        out_dir = "output_curves"
        os.makedirs(out_dir, exist_ok=True)
        out_path = os.path.join(out_dir, os.path.basename(mf4_file).replace(".mf4", "_curve.csv"))
        
        df_curve.to_csv(out_path)
        print(f"✅ 成功儲存輕量化 CSV! (大小: {os.path.getsize(out_path)} bytes)")
        print(f"📁 路徑: {out_path}")
        display(df_curve.head())
        
        mdf.close()
        
    except Exception as e:
        print(f"❌ 錯誤: {e}")

# 我們先隨便挑幾個剛剛印出來的 BMW 變數測試看看
test_signals = [
    'BMWetmc_u_HvsHtrVoltAct',          # 看起來是某種高壓或者加熱器的實際電壓
    'BMWcif_soc_HvsDisp_si',            # 看起來是儀表板顯示的狀態 SoC
    'BMWhvcha_pwr_DcDispPwrCha_si_KW'   # 看起來是 DC 充電的顯示功率 (kW)
]

extract_specific_signals(mf4_file, test_signals)

📦 找到 3 個信號，開始降採樣(Raster=1s)...
✅ 成功儲存輕量化 CSV! (大小: 8650 bytes)
📁 路徑: output_curves/11.28_Cams_UC5_curve.csv


,BMWetmc_u_HvsHtrVoltAct,BMWcif_soc_HvsDisp_si,BMWhvcha_pwr_DcDispPwrCha_si_KW
timestamps,,,
0.0,0.0,36,0.0
1.0,0.0,36,0.0
2.0,0.0,36,0.0
3.0,0.0,36,0.0
4.0,0.0,36,0.0


### 🔑 Step 4: 建立自定義「信號轉譯字典」(Show / DBC Proxy)
既然我們沒有原廠的 `.dbc` 檔案可以把那些十六進制的報文編號 (例如 `0x3F9`) 翻譯成「電壓」、「電流」，或是把剛剛那些天書一樣的 `BMWetmc_u_HvsHtrVoltAct` 轉成人話。

**解決方案：** 我們自己手作一個 `signal_dictionary.csv` 或 JSON 映射檔！
讓 Dashboard 讀進數據後，自動查詢這個我們手刻的「寶典」，把冷冰冰的代碼轉為「HV Battery Voltage (V)」。

### 🗺️ [PLAN模塊] 擴充與模擬全國測試版圖
為了讓下週的 Demo 更具備「大廠格局」，我們在這一步從全國超級大表 (`national_charge_station.csv`) 中，隨機選擇一些一線戰區 (如 北京、上海、深圳、杭州)，「假裝」我們已經把目標排進去了！這樣主管看到的時候，地圖上的數據量會非常驚人！

In [25]:
import pandas as pd
import numpy as np

# 建立用來模擬的全國數據目標
print("🌍 正在讀取全國站點大數據庫 (national_charge_station.csv) ...")
try:
    national_df = pd.read_csv("national_charge_station.csv")
    
    # 我們挑選四個主要城市來模擬排程 (注意：原始數據裡後綴可能有'市'，例如 '北京市')
    target_cities = ['北京市', '上海市', '深圳市', '杭州市']
    
    # 從源頭對這些城市進行過濾
    simulated_cities_df = national_df[national_df['city'].isin(target_cities)].copy()
    
    # 為每個城市隨機抽出 30 個大型充電站作為我們的「模擬規劃」，以擴充 Demo 數據量
    demo_plan_list = []
    
    for city in target_cities:
        city_stations = simulated_cities_df[simulated_cities_df['city'] == city]
        # 盡量挑選大站（有 DC 充電樁或是 CPO 比較大的）作為優先抽樣目標
        # 這裡簡單隨機抽取 30 個站點
        if len(city_stations) > 30:
            sample_df = city_stations.sample(n=30, random_state=42)
        else:
            sample_df = city_stations
        
        demo_plan_list.append(sample_df)
        
    demo_plan_df = pd.concat(demo_plan_list, ignore_index=True)
    
    # 增加幾個用於 Dashboard 顯示友好的欄位（例如經緯度重命名、新增假的預計測試日期）
    demo_plan_df['Target_Date'] = np.random.choice(['2026-05-10', '2026-05-15', '2026-05-20', '2026-06-01'], size=len(demo_plan_df))
    demo_plan_df['Planned_Status'] = 'Pending' # 標記為尚未測試的未來規劃站點
    
    # 儲存一份 demo_plan.csv 讓前端的 Streamlit 讀取顯示全版圖
    demo_plan_df.to_csv('output/demo_national_plan.csv', index=False, encoding='utf-8-sig')
    
    print(f"✅ 成功生成全國模擬測試版圖！總計抽樣了 {len(demo_plan_df)} 個極具代表性的重點測試站點。")
    print("📁 已儲存至: output/demo_national_plan.csv")
    display(demo_plan_df[['city', 'name', 'operator_name', 'Target_Date']].head(10))

except FileNotFoundError:
    print("❌ 找不到 national_charge_station.csv，請確認您的源資料庫是否存在。")

🌍 正在讀取全國站點大數據庫 (national_charge_station.csv) ...
✅ 成功生成全國模擬測試版圖！總計抽樣了 120 個極具代表性的重點測試站點。
📁 已儲存至: output/demo_national_plan.csv


/var/folders/2h/m0b45vn951zbzlgkk5953xd00000gr/T/ipykernel_36593/1528544959.py:7: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  national_df = pd.read_csv("national_charge_station.csv")


,city,name,operator_name,Target_Date
0,北京市,快充 特来电,特来电,2026-05-20
1,北京市,快充/慢充 珠海驿联,珠海驿联,2026-06-01
2,北京市,快充 国家电网,国家电网,2026-06-01
3,北京市,慢充 云快充,云快充,2026-06-01
4,北京市,慢充 利嘉智充,利嘉智充,2026-06-01
5,北京市,快充 小桔充电,小桔充电,2026-05-10
6,北京市,快充 国家电网,国家电网,2026-05-10
7,北京市,快充 国家电网,国家电网,2026-05-20
8,北京市,快充 云快充,云快充,2026-05-15
9,北京市,快充 小桔充电,小桔充电,2026-05-15


## 🚀 衝刺 LOGBOOK 模塊：外勤神器優化
進入 `Mission_Logging.py`，專注於增強手機端的穩定性。目前知道外勤工程師最怕的就是 APP / 網頁在弱網環境的崩潰。